# 33. 仮想売買と約定時刻・価格の照合
出典：FX (3).ipynb、元セルindex [74, 75, 76, 77, 78, 79]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 74
構文状態：valid


In [ ]:
# ============================================================
# PAPER EXECUTION ENGINE v1
#
# Frozen Champion:
#   BASE_PLUS_REGIME
#
# PURPOSE:
#   Frozen Live Inference Engine の上に
#   仮想売買・ポジション・損益管理層を構築する。
#
# Runtime sequence at each 15m boundary:
#
#   1. Exit due position
#   2. Append newly closed 15m bar
#   3. Hardened inference
#   4. BUY / SELL / NO_TRADE
#   5. Execute new entry at boundary price
#   6. Schedule fixed 30-minute exit
#   7. Update logs / equity
#
# IMPORTANT:
#   NO TRAINING
#   NO OPTIMIZATION
#   NO FEATURE CHANGE
#   NO PARAMETER CHANGE
#   NO REAL ORDERS
# ============================================================

from pathlib import Path
import json
import math
import copy

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 280)


# ============================================================
# 0. REQUIRED PRODUCTION STATE
# ============================================================

REQUIRED_OBJECTS = [

    # Hardened Production runtime
    "PRODUCTION_CANONICAL_HISTORY",
    "PRODUCTION_RUNTIME_CONTRACT",
    "PRODUCTION_HARDENING_REGRESSION_OK",

    # Hardened functions
    "validate_canonical_history",
    "normalize_single_closed_bar",
    "append_closed_bar_to_canonical_history",
    "run_hardened_live_inference",

    # Champion info
    "LIVE_MANIFEST",
    "LIVE_FEATURES",
    "LIVE_CHAMPION_DIR",
]


missing_objects = [

    name
    for name in REQUIRED_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Paper Execution Engineに必要な"
        "Production状態が不足しています。\n\n"

        f"Missing:\n{missing_objects}\n\n"

        "Live Engine Hardeningセルまで"
        "先に実行してください。"
    )


if not bool(
    PRODUCTION_HARDENING_REGRESSION_OK
):

    raise RuntimeError(

        "Production Live Engine Hardeningが"
        "PASSしていません。"
    )


# ============================================================
# 1. FROZEN EXECUTION SPEC
# ============================================================

PAPER_CONTRACT = copy.deepcopy(
    PRODUCTION_RUNTIME_CONTRACT
)


EXPECTED_CHAMPION = (
    "BASE_PLUS_REGIME"
)


if (
    PAPER_CONTRACT[
        "champion_name"
    ]
    !=
    EXPECTED_CHAMPION
):

    raise RuntimeError(

        "Wrong Champion.\n"

        f"Expected: {EXPECTED_CHAMPION}\n"

        f"Actual: "
        f"{PAPER_CONTRACT['champion_name']}"
    )


if int(
    PAPER_CONTRACT[
        "bar_interval_minutes"
    ]
) != 15:

    raise RuntimeError(
        "Bar interval must be 15 minutes."
    )


if int(
    PAPER_CONTRACT[
        "holding_period_minutes"
    ]
) != 30:

    raise RuntimeError(
        "Holding period must be 30 minutes."
    )


if bool(
    PAPER_CONTRACT[
        "overlapping_positions"
    ]
):

    raise RuntimeError(
        "Overlapping positions must be prohibited."
    )


if (
    PAPER_CONTRACT[
        "runtime_history_policy"
    ]
    !=
    "CANONICAL_FULL_HISTORY_REQUIRED"
):

    raise RuntimeError(
        "Canonical full-history policy mismatch."
    )


PAPER_BASE_COST = float(

    PAPER_CONTRACT[
        "cost_per_trade_return"
    ]
)


PAPER_HOLD_MINUTES = int(

    PAPER_CONTRACT[
        "holding_period_minutes"
    ]
)


# ============================================================
# 2. HELPERS
# ============================================================

def paper_to_utc_timestamp(
    value
):

    ts = pd.Timestamp(
        value
    )

    if ts.tzinfo is None:

        ts = ts.tz_localize(
            "UTC"
        )

    else:

        ts = ts.tz_convert(
            "UTC"
        )

    return ts


def paper_validate_price(
    price,
    name="price"
):

    try:

        value = float(
            price
        )

    except Exception as exc:

        raise RuntimeError(
            f"{name} is not numeric: {price}"
        ) from exc


    if not np.isfinite(
        value
    ):

        raise RuntimeError(
            f"{name} is not finite."
        )


    if value <= 0:

        raise RuntimeError(
            f"{name} must be > 0."
        )


    return value


# ============================================================
# 3. RETURN ACCOUNTING
#
# Must reproduce research convention:
#
# BUY:
#   gross = exit / entry - 1
#
# SELL:
#   gross = -(exit / entry - 1)
#
# net:
#   position_size * (gross - cost)
#
# ============================================================

def paper_trade_return(

    side,

    entry_price,

    exit_price,

    position_size,

    cost=PAPER_BASE_COST
):

    side = str(
        side
    ).upper()


    if side not in [
        "BUY",
        "SELL",
    ]:

        raise RuntimeError(
            f"Invalid trade side: {side}"
        )


    entry_price = paper_validate_price(
        entry_price,
        "entry_price"
    )


    exit_price = paper_validate_price(
        exit_price,
        "exit_price"
    )


    position_size = float(
        position_size
    )


    if (

        not np.isfinite(
            position_size
        )

        or

        position_size <= 0

    ):

        raise RuntimeError(
            "position_size must be positive."
        )


    long_return = (

        exit_price

        /

        entry_price

        -

        1.0
    )


    if side == "BUY":

        gross_return = (
            long_return
        )

    else:

        gross_return = (
            -long_return
        )


    net_return = (

        position_size

        *

        (
            gross_return

            -

            float(
                cost
            )
        )
    )


    return {

        "gross_return":
            float(
                gross_return
            ),

        "net_return":
            float(
                net_return
            ),
    }


# ============================================================
# 4. PAPER EXECUTION ENGINE
# ============================================================

class PaperExecutionEngine:

    def __init__(

        self,

        canonical_history,

        starting_equity=1_000_000.0,

        strict_schedule=True
    ):

        # ----------------------------------------------------
        # Validate starting equity
        # ----------------------------------------------------

        self.starting_equity = float(
            starting_equity
        )


        if (

            not np.isfinite(
                self.starting_equity
            )

            or

            self.starting_equity <= 0

        ):

            raise RuntimeError(
                "starting_equity must be > 0."
            )


        self.equity = float(
            self.starting_equity
        )


        self.strict_schedule = bool(
            strict_schedule
        )


        # ----------------------------------------------------
        # Canonical Production history
        # ----------------------------------------------------

        checked_history, report = (

            validate_canonical_history(
                canonical_history
            )
        )


        self.history = (
            checked_history.copy()
        )


        self.initial_history_report = (
            report.copy()
        )


        # ----------------------------------------------------
        # Trading state
        # ----------------------------------------------------

        self.open_position = None

        self.pending_order = None


        # ----------------------------------------------------
        # Safety
        # ----------------------------------------------------

        self.safety_halted = False

        self.safety_reason = None


        # ----------------------------------------------------
        # Deduplication
        # ----------------------------------------------------

        self.processed_boundaries = set()

        self.processed_signal_times = set()


        # ----------------------------------------------------
        # Logs
        # ----------------------------------------------------

        self.trade_log = []

        self.event_log = []


        # ----------------------------------------------------
        # Counters
        # ----------------------------------------------------

        self.boundaries_processed = 0

        self.signals_seen = 0

        self.no_trade_signals = 0

        self.entries = 0

        self.exits = 0


        self._log_event(

            event_type="ENGINE_INITIALIZED",

            timestamp=self.history.index[-1],

            details={

                "starting_equity":
                    self.starting_equity,

                "canonical_rows":
                    len(
                        self.history
                    ),

                "canonical_latest":
                    str(
                        self.history.index[-1]
                    ),

                "champion":
                    PAPER_CONTRACT[
                        "champion_name"
                    ],

                "champion_version":
                    PAPER_CONTRACT[
                        "champion_version"
                    ],
            }
        )


    # ========================================================
    # INTERNAL LOGGING
    # ========================================================

    def _log_event(

        self,

        event_type,

        timestamp,

        details=None
    ):

        if details is None:

            details = {}


        row = {

            "event_id":
                len(
                    self.event_log
                )
                +
                1,

            "timestamp":
                paper_to_utc_timestamp(
                    timestamp
                ),

            "event_type":
                str(
                    event_type
                ),

            "details":
                copy.deepcopy(
                    details
                ),
        }


        self.event_log.append(
            row
        )


    # ========================================================
    # SAFETY HALT
    # ========================================================

    def _halt(

        self,

        reason,

        timestamp=None
    ):

        self.safety_halted = True

        self.safety_reason = str(
            reason
        )


        if timestamp is None:

            timestamp = (
                self.history.index[-1]
            )


        self._log_event(

            event_type="SAFETY_HALT",

            timestamp=timestamp,

            details={
                "reason":
                    self.safety_reason
            }
        )


        raise RuntimeError(

            "PAPER ENGINE SAFETY HALT\n\n"

            f"{self.safety_reason}"
        )


    # ========================================================
    # STATE
    # ========================================================

    def status(
        self
    ):

        return {

            "equity":
                float(
                    self.equity
                ),

            "starting_equity":
                float(
                    self.starting_equity
                ),

            "total_return":
                float(

                    self.equity

                    /

                    self.starting_equity

                    -

                    1.0
                ),

            "open_position":
                copy.deepcopy(
                    self.open_position
                ),

            "pending_order":
                copy.deepcopy(
                    self.pending_order
                ),

            "canonical_rows":
                int(
                    len(
                        self.history
                    )
                ),

            "canonical_latest":
                self.history.index[-1],

            "boundaries_processed":
                int(
                    self.boundaries_processed
                ),

            "signals_seen":
                int(
                    self.signals_seen
                ),

            "no_trade_signals":
                int(
                    self.no_trade_signals
                ),

            "entries":
                int(
                    self.entries
                ),

            "exits":
                int(
                    self.exits
                ),

            "safety_halted":
                bool(
                    self.safety_halted
                ),

            "safety_reason":
                self.safety_reason,
        }


    # ========================================================
    # CLOSE POSITION
    #
    # Called BEFORE inference at a boundary.
    # ========================================================

    def _close_due_position(

        self,

        boundary_time,

        boundary_price
    ):

        if self.open_position is None:

            return None


        boundary_time = (

            paper_to_utc_timestamp(
                boundary_time
            )
        )


        boundary_price = (

            paper_validate_price(
                boundary_price,
                "boundary_price"
            )
        )


        exit_time = (

            paper_to_utc_timestamp(

                self.open_position[
                    "planned_exit_time"
                ]
            )
        )


        # ----------------------------------------------------
        # Not due yet
        # ----------------------------------------------------

        if (
            boundary_time
            <
            exit_time
        ):

            return None


        # ----------------------------------------------------
        # Missed exact exit time
        # ----------------------------------------------------

        if (

            boundary_time
            >
            exit_time

            and

            self.strict_schedule

        ):

            self._halt(

                reason=(

                    "Scheduled exit was missed.\n"

                    f"Expected exit: {exit_time}\n"

                    f"Current boundary: {boundary_time}"
                ),

                timestamp=boundary_time
            )


        position = copy.deepcopy(
            self.open_position
        )


        returns = paper_trade_return(

            side=
                position[
                    "side"
                ],

            entry_price=
                position[
                    "entry_price"
                ],

            exit_price=
                boundary_price,

            position_size=
                position[
                    "position_size"
                ],

            cost=
                PAPER_BASE_COST
        )


        equity_before = float(
            self.equity
        )


        self.equity = (

            self.equity

            *

            (
                1.0

                +

                returns[
                    "net_return"
                ]
            )
        )


        equity_after = float(
            self.equity
        )


        trade_id = int(
            position[
                "trade_id"
            ]
        )


        trade_record = {

            "trade_id":
                trade_id,

            "signal_time":
                position[
                    "signal_time"
                ],

            "side":
                position[
                    "side"
                ],

            "confidence":
                position[
                    "confidence"
                ],

            "p_up":
                position[
                    "p_up"
                ],

            "position_size":
                position[
                    "position_size"
                ],

            "entry_time":
                position[
                    "entry_time"
                ],

            "entry_price":
                position[
                    "entry_price"
                ],

            "planned_exit_time":
                position[
                    "planned_exit_time"
                ],

            "actual_exit_time":
                boundary_time,

            "exit_price":
                boundary_price,

            "gross_return":
                returns[
                    "gross_return"
                ],

            "cost":
                PAPER_BASE_COST,

            "net_return":
                returns[
                    "net_return"
                ],

            "equity_before":
                equity_before,

            "equity_after":
                equity_after,
        }


        self.trade_log.append(
            trade_record
        )


        self.exits += 1


        self._log_event(

            event_type="POSITION_CLOSED",

            timestamp=boundary_time,

            details={
                "trade_id":
                    trade_id,

                "side":
                    position[
                        "side"
                    ],

                "exit_price":
                    boundary_price,

                "gross_return":
                    returns[
                        "gross_return"
                    ],

                "net_return":
                    returns[
                        "net_return"
                    ],

                "equity_after":
                    equity_after,
            }
        )


        self.open_position = None


        return trade_record


    # ========================================================
    # CREATE ORDER FROM FROZEN SIGNAL
    # ========================================================

    def _create_pending_order(

        self,

        inference_result,

        boundary_time
    ):

        action = str(

            inference_result[
                "action"
            ]
        ).upper()


        if action not in [
            "BUY",
            "SELL",
        ]:

            return None


        signal_time = (

            paper_to_utc_timestamp(

                inference_result[
                    "signal_time"
                ]
            )
        )


        if signal_time in (
            self.processed_signal_times
        ):

            self._halt(

                reason=(

                    "Duplicate trade signal detected.\n"

                    f"Signal time: {signal_time}"
                ),

                timestamp=boundary_time
            )


        planned_entry_time = (

            paper_to_utc_timestamp(

                inference_result[
                    "planned_entry_time"
                ]
            )
        )


        planned_exit_time = (

            paper_to_utc_timestamp(

                inference_result[
                    "planned_exit_time"
                ]
            )
        )


        boundary_time = (

            paper_to_utc_timestamp(
                boundary_time
            )
        )


        if (
            planned_entry_time
            !=
            boundary_time
        ):

            self._halt(

                reason=(

                    "Entry schedule mismatch.\n"

                    f"Planned entry: {planned_entry_time}\n"

                    f"Boundary:      {boundary_time}"
                ),

                timestamp=boundary_time
            )


        expected_exit = (

            planned_entry_time

            +

            pd.Timedelta(
                minutes=PAPER_HOLD_MINUTES
            )
        )


        if (
            planned_exit_time
            !=
            expected_exit
        ):

            self._halt(

                reason=(

                    "30-minute exit contract violation.\n"

                    f"Expected: {expected_exit}\n"

                    f"Actual:   {planned_exit_time}"
                ),

                timestamp=boundary_time
            )


        if self.open_position is not None:

            self._halt(

                reason=(
                    "New order attempted while position is open."
                ),

                timestamp=boundary_time
            )


        order = {

            "signal_time":
                signal_time,

            "side":
                action,

            "confidence":
                float(
                    inference_result[
                        "confidence"
                    ]
                ),

            "p_up":
                float(
                    inference_result[
                        "calibrated_p_up"
                    ]
                ),

            "position_size":
                float(
                    inference_result[
                        "position_size"
                    ]
                ),

            "planned_entry_time":
                planned_entry_time,

            "planned_exit_time":
                planned_exit_time,
        }


        self.pending_order = (
            order
        )


        self.processed_signal_times.add(
            signal_time
        )


        self._log_event(

            event_type="ORDER_CREATED",

            timestamp=boundary_time,

            details=copy.deepcopy(
                order
            )
        )


        return order


    # ========================================================
    # EXECUTE ENTRY
    # ========================================================

    def _execute_pending_entry(

        self,

        boundary_time,

        boundary_price
    ):

        if self.pending_order is None:

            return None


        boundary_time = (

            paper_to_utc_timestamp(
                boundary_time
            )
        )


        boundary_price = (

            paper_validate_price(
                boundary_price,
                "boundary_price"
            )
        )


        entry_time = (

            paper_to_utc_timestamp(

                self.pending_order[
                    "planned_entry_time"
                ]
            )
        )


        if (
            boundary_time
            <
            entry_time
        ):

            return None


        if (
            boundary_time
            >
            entry_time
        ):

            # Never enter late.
            order = copy.deepcopy(
                self.pending_order
            )

            self.pending_order = None


            self._log_event(

                event_type="MISSED_ENTRY_CANCELLED",

                timestamp=boundary_time,

                details={
                    "expected_entry":
                        entry_time,

                    "actual_boundary":
                        boundary_time,

                    "signal_time":
                        order[
                            "signal_time"
                        ],
                }
            )


            if self.strict_schedule:

                self._halt(

                    reason=(

                        "Scheduled entry was missed.\n"

                        f"Expected: {entry_time}\n"

                        f"Current:  {boundary_time}"
                    ),

                    timestamp=boundary_time
                )


            return None


        if self.open_position is not None:

            self._halt(

                reason=(
                    "Overlap protection triggered at entry."
                ),

                timestamp=boundary_time
            )


        order = copy.deepcopy(
            self.pending_order
        )


        trade_id = (
            self.entries
            +
            1
        )


        position = {

            "trade_id":
                trade_id,

            "signal_time":
                order[
                    "signal_time"
                ],

            "side":
                order[
                    "side"
                ],

            "confidence":
                order[
                    "confidence"
                ],

            "p_up":
                order[
                    "p_up"
                ],

            "position_size":
                order[
                    "position_size"
                ],

            "entry_time":
                boundary_time,

            "entry_price":
                boundary_price,

            "planned_exit_time":
                order[
                    "planned_exit_time"
                ],
        }


        self.open_position = (
            position
        )


        self.pending_order = None


        self.entries += 1


        self._log_event(

            event_type="POSITION_OPENED",

            timestamp=boundary_time,

            details=copy.deepcopy(
                position
            )
        )


        return copy.deepcopy(
            position
        )


    # ========================================================
    # PROCESS ONE 15M BOUNDARY
    #
    # closed_bar:
    #   bar whose timestamp is boundary_time - 15m
    #
    # boundary_price:
    #   exact open / executable price at boundary_time
    #
    # Example:
    #
    # closed bar timestamp = 10:00
    # boundary_time        = 10:15
    # boundary_price       = 10:15 bar OPEN
    # ========================================================

    def process_boundary(

        self,

        closed_bar,

        boundary_time,

        boundary_price,

        as_of_utc=None,

        allow_verified_gap=False
    ):

        if self.safety_halted:

            raise RuntimeError(

                "Paper Engine is halted.\n"

                f"Reason: {self.safety_reason}"
            )


        boundary_time = (

            paper_to_utc_timestamp(
                boundary_time
            )
        )


        boundary_price = (

            paper_validate_price(
                boundary_price,
                "boundary_price"
            )
        )


        if as_of_utc is None:

            as_of_utc = (
                boundary_time
            )


        else:

            as_of_utc = (

                paper_to_utc_timestamp(
                    as_of_utc
                )
            )


        # ----------------------------------------------------
        # Duplicate boundary protection
        # ----------------------------------------------------

        if (
            boundary_time
            in
            self.processed_boundaries
        ):

            self._halt(

                reason=(

                    "Duplicate boundary detected.\n"

                    f"Boundary: {boundary_time}"
                ),

                timestamp=boundary_time
            )


        # ----------------------------------------------------
        # Normalize incoming closed bar
        # ----------------------------------------------------

        normalized_bar = (

            normalize_single_closed_bar(
                closed_bar
            )
        )


        closed_bar_time = (
            normalized_bar.index[0]
        )


        expected_boundary = (

            closed_bar_time

            +

            pd.Timedelta(
                minutes=15
            )
        )


        if (
            boundary_time
            !=
            expected_boundary
        ):

            self._halt(

                reason=(

                    "Closed bar / boundary timing mismatch.\n"

                    f"Closed bar open: {closed_bar_time}\n"

                    f"Expected boundary: {expected_boundary}\n"

                    f"Actual boundary:   {boundary_time}"
                ),

                timestamp=boundary_time
            )


        # ----------------------------------------------------
        # STEP 1
        # EXIT DUE POSITION FIRST
        # ----------------------------------------------------

        closed_trade = (

            self._close_due_position(

                boundary_time=
                    boundary_time,

                boundary_price=
                    boundary_price
            )
        )


        # ----------------------------------------------------
        # STEP 2
        # APPEND CLOSED BAR TO CANONICAL HISTORY
        # ----------------------------------------------------

        try:

            updated_history, append_report = (

                append_closed_bar_to_canonical_history(

                    canonical_history=
                        self.history,

                    new_bar=
                        normalized_bar,

                    as_of_utc=
                        as_of_utc,

                    allow_verified_gap=
                        allow_verified_gap
                )
            )

        except Exception as exc:

            self._halt(

                reason=(

                    "Canonical history append failed.\n"

                    f"{type(exc).__name__}: {exc}"
                ),

                timestamp=boundary_time
            )


        self.history = (
            updated_history
        )


        # ----------------------------------------------------
        # STEP 3
        # HARDENED FROZEN INFERENCE
        # ----------------------------------------------------

        inference = (

            run_hardened_live_inference(

                canonical_history=
                    self.history,

                position_is_open=(
                    self.open_position
                    is not None
                ),

                as_of_utc=
                    as_of_utc
            )
        )


        self.signals_seen += 1


        # ----------------------------------------------------
        # Hard timing validation
        # ----------------------------------------------------

        inference_signal_time = (

            paper_to_utc_timestamp(

                inference[
                    "signal_time"
                ]
            )
        )


        if (
            inference_signal_time
            !=
            closed_bar_time
        ):

            self._halt(

                reason=(

                    "Inference signal timestamp mismatch.\n"

                    f"Expected: {closed_bar_time}\n"

                    f"Actual:   {inference_signal_time}"
                ),

                timestamp=boundary_time
            )


        # ----------------------------------------------------
        # STEP 4
        # SIGNAL
        # ----------------------------------------------------

        action = str(

            inference[
                "action"
            ]
        ).upper()


        opened_position = None


        if action in [
            "BUY",
            "SELL",
        ]:

            self._create_pending_order(

                inference_result=
                    inference,

                boundary_time=
                    boundary_time
            )


            # ------------------------------------------------
            # STEP 5
            # Execute at exact same 15m boundary
            # ------------------------------------------------

            opened_position = (

                self._execute_pending_entry(

                    boundary_time=
                        boundary_time,

                    boundary_price=
                        boundary_price
                )
            )


        else:

            self.no_trade_signals += 1


            self._log_event(

                event_type="NO_TRADE",

                timestamp=boundary_time,

                details={

                    "signal_time":
                        inference_signal_time,

                    "reason":
                        inference[
                            "reason"
                        ],

                    "confidence":
                        float(
                            inference[
                                "confidence"
                            ]
                        ),

                    "threshold":
                        float(
                            inference[
                                "threshold"
                            ]
                        ),
                }
            )


        # ----------------------------------------------------
        # STEP 6
        # Mark boundary complete
        # ----------------------------------------------------

        self.processed_boundaries.add(
            boundary_time
        )


        self.boundaries_processed += 1


        result = {

            "boundary_time":
                boundary_time,

            "closed_bar_time":
                closed_bar_time,

            "boundary_price":
                boundary_price,

            "closed_trade":
                closed_trade,

            "inference":
                copy.deepcopy(
                    inference
                ),

            "opened_position":
                opened_position,

            "equity":
                float(
                    self.equity
                ),

            "position_open":
                bool(
                    self.open_position
                    is not None
                ),

            "canonical_rows":
                int(
                    len(
                        self.history
                    )
                ),
        }


        return result


    # ========================================================
    # DATAFRAMES
    # ========================================================

    def trades_dataframe(
        self
    ):

        if not self.trade_log:

            return pd.DataFrame(

                columns=[

                    "trade_id",
                    "signal_time",
                    "side",
                    "confidence",
                    "p_up",
                    "position_size",

                    "entry_time",
                    "entry_price",

                    "planned_exit_time",
                    "actual_exit_time",
                    "exit_price",

                    "gross_return",
                    "cost",
                    "net_return",

                    "equity_before",
                    "equity_after",
                ]
            )


        df = pd.DataFrame(
            self.trade_log
        )


        return (
            df.sort_values(
                "trade_id"
            )
            .reset_index(
                drop=True
            )
        )


    def events_dataframe(
        self
    ):

        if not self.event_log:

            return pd.DataFrame(
                columns=[
                    "event_id",
                    "timestamp",
                    "event_type",
                    "details",
                ]
            )


        return (

            pd.DataFrame(
                self.event_log
            )

            .sort_values(
                "event_id"
            )

            .reset_index(
                drop=True
            )
        )


# ============================================================
# 5. UNIT TEST - RETURN ACCOUNTING
# ============================================================

print("=" * 110)
print("PAPER ENGINE MECHANICS UNIT TEST")
print("=" * 110)


buy_test = paper_trade_return(

    side="BUY",

    entry_price=100.0,

    exit_price=101.0,

    position_size=1.0,

    cost=PAPER_BASE_COST
)


sell_test = paper_trade_return(

    side="SELL",

    entry_price=100.0,

    exit_price=99.0,

    position_size=1.0,

    cost=PAPER_BASE_COST
)


expected_buy = (

    0.01
    -
    PAPER_BASE_COST
)


expected_sell = (

    0.01
    -
    PAPER_BASE_COST
)


BUY_RETURN_TEST_OK = np.isclose(

    buy_test[
        "net_return"
    ],

    expected_buy,

    atol=1e-12,

    rtol=0.0
)


SELL_RETURN_TEST_OK = np.isclose(

    sell_test[
        "net_return"
    ],

    expected_sell,

    atol=1e-12,

    rtol=0.0
)


print(
    "BUY accounting:",
    BUY_RETURN_TEST_OK
)

print(
    "SELL accounting:",
    SELL_RETURN_TEST_OK
)


if not (

    BUY_RETURN_TEST_OK

    and

    SELL_RETURN_TEST_OK
):

    raise RuntimeError(
        "Paper return accounting unit test failed."
    )


# ============================================================
# 6. CREATE PAPER ENGINE
#
# NOTE:
# We do NOT process a new bar here.
# Current history remains unchanged.
# ============================================================

PAPER_ENGINE = PaperExecutionEngine(

    canonical_history=
        PRODUCTION_CANONICAL_HISTORY,

    starting_equity=
        1_000_000.0,

    strict_schedule=
        True
)


# ============================================================
# 7. INITIAL STATE CHECK
# ============================================================

PAPER_ENGINE_INITIAL_STATUS = (
    PAPER_ENGINE.status()
)


INITIAL_STATE_CHECKS = {

    "not_halted":

        not PAPER_ENGINE_INITIAL_STATUS[
            "safety_halted"
        ],

    "flat":

        PAPER_ENGINE_INITIAL_STATUS[
            "open_position"
        ]
        is None,

    "no_pending_order":

        PAPER_ENGINE_INITIAL_STATUS[
            "pending_order"
        ]
        is None,

    "no_entries":

        PAPER_ENGINE_INITIAL_STATUS[
            "entries"
        ]
        ==
        0,

    "no_exits":

        PAPER_ENGINE_INITIAL_STATUS[
            "exits"
        ]
        ==
        0,

    "equity_intact":

        np.isclose(

            PAPER_ENGINE_INITIAL_STATUS[
                "equity"
            ],

            1_000_000.0,

            atol=1e-9,
            rtol=0.0
        ),
}


INITIAL_STATE_OK = all(
    INITIAL_STATE_CHECKS.values()
)


print()
print("=" * 110)
print("INITIAL PAPER ENGINE STATE")
print("=" * 110)


for key, value in (
    INITIAL_STATE_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


print()

print(
    "INITIAL STATE PASSED:",
    INITIAL_STATE_OK
)


if not INITIAL_STATE_OK:

    raise RuntimeError(
        "Paper Engine initial state failed."
    )


# ============================================================
# 8. CURRENT HISTORY PREVIEW
#
# This MUST NOT modify Paper Engine state.
# It simply confirms that the engine is connected to the
# same Hardened Live Inference pipeline.
# ============================================================

PAPER_PREVIEW_RESULT = (

    run_hardened_live_inference(

        canonical_history=
            PAPER_ENGINE.history,

        position_is_open=False,

        as_of_utc=None
    )
)


PREVIEW_CHECKS = {

    "canonical_history":

        bool(
            PAPER_PREVIEW_RESULT[
                "canonical_history_ok"
            ]
        ),

    "frozen_hash":

        bool(
            PAPER_PREVIEW_RESULT[
                "frozen_prefix_hash_ok"
            ]
        ),

    "runtime_mode":

        PAPER_PREVIEW_RESULT[
            "runtime_mode"
        ]

        ==
        "CANONICAL_FULL_HISTORY",

    "feature_count":

        int(
            PAPER_PREVIEW_RESULT[
                "feature_count"
            ]
        )

        ==
        41,
}


PREVIEW_OK = all(
    PREVIEW_CHECKS.values()
)


print()
print("=" * 110)
print("PAPER -> LIVE ENGINE CONNECTION TEST")
print("=" * 110)


for key, value in (
    PREVIEW_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


print()

print(
    "Latest signal time:",
    PAPER_PREVIEW_RESULT[
        "signal_time"
    ]
)

print(
    "Latest action:",
    PAPER_PREVIEW_RESULT[
        "action"
    ]
)

print(
    "Latest reason:",
    PAPER_PREVIEW_RESULT[
        "reason"
    ]
)

print(
    "Latest confidence:",
    PAPER_PREVIEW_RESULT[
        "confidence"
    ]
)


if not PREVIEW_OK:

    raise RuntimeError(
        "Paper -> Live Engine connection failed."
    )


# ============================================================
# 9. PAPER ENGINE CONTRACT
# ============================================================

PAPER_ENGINE_CONTRACT = {

    "schema_version":
        1,

    "champion_name":
        PAPER_CONTRACT[
            "champion_name"
        ],

    "champion_version":
        PAPER_CONTRACT[
            "champion_version"
        ],

    "execution_mode":
        "PAPER_ONLY",

    "real_orders_allowed":
        False,

    "bar_interval_minutes":
        15,

    "boundary_sequence": [

        "EXIT_DUE_POSITION",

        "APPEND_CLOSED_BAR",

        "HARDENED_INFERENCE",

        "CREATE_ORDER",

        "EXECUTE_ENTRY",
    ],

    "entry_policy":
        "NEXT_15M_BAR_OPEN",

    "exit_policy":
        "FIXED_30_MINUTES",

    "holding_period_minutes":
        PAPER_HOLD_MINUTES,

    "overlap_allowed":
        False,

    "late_entry_allowed":
        False,

    "strict_exit_schedule":
        True,

    "history_policy":
        "CANONICAL_FULL_HISTORY_REQUIRED",

    "historical_overwrite_allowed":
        False,

    "duplicate_boundary_allowed":
        False,

    "base_cost_return":
        PAPER_BASE_COST,

    "return_accounting":

        "position_size * "
        "(directional_gross_return - base_cost)",

    "equity_accounting":
        "COMPOUNDED",

    "starting_equity":
        PAPER_ENGINE.starting_equity,
}


# ============================================================
# 10. SAVE PAPER ENGINE CONTRACT
# ============================================================

PAPER_RUNTIME_DIR = (

    Path(
        LIVE_CHAMPION_DIR
    )

    /

    "runtime"

    /

    "paper"
)


PAPER_RUNTIME_DIR.mkdir(

    parents=True,

    exist_ok=True
)


PAPER_ENGINE_CONTRACT_PATH = (

    PAPER_RUNTIME_DIR

    /

    "paper_execution_contract.json"
)


with open(

    PAPER_ENGINE_CONTRACT_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        PAPER_ENGINE_CONTRACT,

        f,

        ensure_ascii=False,

        indent=2
    )


# ============================================================
# 11. FINAL BUILD CHECKS
# ============================================================

PAPER_ENGINE_BUILD_CHECKS = {

    "live_hardening_passed":

        bool(
            PRODUCTION_HARDENING_REGRESSION_OK
        ),

    "buy_return_accounting":

        bool(
            BUY_RETURN_TEST_OK
        ),

    "sell_return_accounting":

        bool(
            SELL_RETURN_TEST_OK
        ),

    "engine_initial_state":

        bool(
            INITIAL_STATE_OK
        ),

    "live_engine_connection":

        bool(
            PREVIEW_OK
        ),

    "flat_start":

        PAPER_ENGINE.open_position
        is None,

    "no_pending_order":

        PAPER_ENGINE.pending_order
        is None,

    "paper_only":

        PAPER_ENGINE_CONTRACT[
            "real_orders_allowed"
        ]
        is False,

    "overlap_prohibited":

        PAPER_ENGINE_CONTRACT[
            "overlap_allowed"
        ]
        is False,

    "fixed_30m_exit":

        PAPER_ENGINE_CONTRACT[
            "holding_period_minutes"
        ]
        ==
        30,

    "canonical_history_required":

        PAPER_ENGINE_CONTRACT[
            "history_policy"
        ]
        ==
        "CANONICAL_FULL_HISTORY_REQUIRED",

    "contract_saved":

        PAPER_ENGINE_CONTRACT_PATH.exists(),
}


PAPER_ENGINE_BUILD_OK = all(

    PAPER_ENGINE_BUILD_CHECKS.values()
)


if PAPER_ENGINE_BUILD_OK:

    PAPER_ENGINE_BUILD_DECISION = (

        "PASS_READY_FOR_EXECUTION_REPLAY_TEST"
    )

else:

    PAPER_ENGINE_BUILD_DECISION = (

        "FAIL_DO_NOT_RUN_PAPER_EXECUTION"
    )


# ============================================================
# 12. SAVE NOTEBOOK VARIABLES
# ============================================================

PRODUCTION_PAPER_ENGINE = (
    PAPER_ENGINE
)


PRODUCTION_PAPER_ENGINE_CONTRACT = (
    PAPER_ENGINE_CONTRACT.copy()
)


PRODUCTION_PAPER_ENGINE_CONTRACT_PATH = str(
    PAPER_ENGINE_CONTRACT_PATH
)


PRODUCTION_PAPER_ENGINE_BUILD_CHECKS = (
    PAPER_ENGINE_BUILD_CHECKS.copy()
)


PRODUCTION_PAPER_ENGINE_BUILD_OK = (
    PAPER_ENGINE_BUILD_OK
)


PRODUCTION_PAPER_ENGINE_DECISION = (
    PAPER_ENGINE_BUILD_DECISION
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print()
print("=" * 110)
print("PAPER EXECUTION ENGINE BUILD REPORT")
print("=" * 110)


print(
    "Champion:",
    PAPER_CONTRACT[
        "champion_name"
    ]
)

print(
    "Version:",
    PAPER_CONTRACT[
        "champion_version"
    ]
)

print(
    "Starting equity:",
    f"{PAPER_ENGINE.starting_equity:,.2f}"
)

print(
    "Current equity:",
    f"{PAPER_ENGINE.equity:,.2f}"
)

print(
    "Canonical rows:",
    f"{len(PAPER_ENGINE.history):,}"
)

print(
    "Canonical latest:",
    PAPER_ENGINE.history.index[-1]
)

print(
    "Cost:",
    PAPER_BASE_COST
)

print(
    "Holding period:",
    PAPER_HOLD_MINUTES,
    "minutes"
)

print(
    "Open position:",
    PAPER_ENGINE.open_position
)

print(
    "Pending order:",
    PAPER_ENGINE.pending_order
)


print()
print(
    "Paper contract:"
)

print(
    PAPER_ENGINE_CONTRACT_PATH
)


print()
print("=" * 110)
print("FINAL CHECKS")
print("=" * 110)


for key, value in (
    PAPER_ENGINE_BUILD_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


print()

print(
    "PAPER EXECUTION ENGINE BUILD PASSED:",
    PAPER_ENGINE_BUILD_OK
)


print()

print(
    "FINAL DECISION:",
    PAPER_ENGINE_BUILD_DECISION
)


if PAPER_ENGINE_BUILD_OK:

    print()

    print(
        "Paper Execution Engine is constructed successfully."
    )

    print(
        "No real orders can be sent."
    )

    print(
        "The Frozen Champion has not been modified."
    )

    print()

    print(
        "NEXT STEP:"
    )

    print(
        "2026 HISTORICAL PAPER EXECUTION REPLAY TEST."
    )

    print()

    print(
        "The next test will feed historical 15m boundaries"
    )

    print(
        "through this exact PaperExecutionEngine and verify:"
    )

    print(
        "300 expected trades,"
    )

    print(
        "300 executed trades,"
    )

    print(
        "0 missing,"
    )

    print(
        "0 unexpected,"
    )

    print(
        "0 duplicate,"
    )

    print(
        "0 overlap,"
    )

    print(
        "and matching entry/exit/return/equity behavior."
    )


else:

    print()

    print(
        "STOP."
    )

    print(
        "Do not start Execution Replay or Paper Trading."
    )


## 元セルindex 75
構文状態：valid


In [ ]:
# ============================================================
# 2026 HISTORICAL PAPER EXECUTION REPLAY TEST v1
#
# PURPOSE
# ------------------------------------------------------------
# Historically-valid 2026 Frozen Champion signalsを使い、
# Execution layerだけを厳密に再現する。
#
# Checks:
#   - Expected trades = 300
#   - Executed trades = 300
#   - Missing = 0
#   - Unexpected = 0
#   - Duplicate = 0
#   - Overlap = 0
#   - Side
#   - Position size
#   - Entry time
#   - Exit time
#   - Entry price
#   - Exit price
#   - Gross return
#   - Net return
#   - PF
#   - Avg return
#   - Growth
#   - Max DD
#
# IMPORTANT
# ------------------------------------------------------------
# NO TRAINING
# NO OPTIMIZATION
# NO FEATURE CHANGE
# NO PRODUCTION MODEL REPLAY ON 2026
# NO REAL ORDERS
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 300)


# ============================================================
# 0. PRECHECK
# ============================================================

REQUIRED = [
    "PRODUCTION_PAPER_ENGINE_BUILD_OK",
    "PRODUCTION_PAPER_ENGINE_CONTRACT",
    "LIVE_CHAMPION_DIR",
    "paper_trade_return",
]

missing = [
    name for name in REQUIRED
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "必要なPaper Engine状態が不足しています。\n\n"
        f"Missing: {missing}\n\n"
        "Paper Execution Engine Buildセルを先に実行してください。"
    )


if not bool(PRODUCTION_PAPER_ENGINE_BUILD_OK):
    raise RuntimeError(
        "Paper Execution Engine BuildがPASSしていません。"
    )


EXPECTED_TRADE_COUNT = 300

TEST_YEAR = 2026

PRICE_TOL = 1e-10
RETURN_TOL = 1e-12
SIZE_TOL = 1e-12

BASE_COST = float(
    PRODUCTION_PAPER_ENGINE_CONTRACT[
        "base_cost_return"
    ]
)

HOLD_MINUTES = int(
    PRODUCTION_PAPER_ENGINE_CONTRACT[
        "holding_period_minutes"
    ]
)


if HOLD_MINUTES != 30:
    raise RuntimeError(
        f"Frozen holding period must be 30m. Actual={HOLD_MINUTES}"
    )


# ============================================================
# 1. COLUMN HELPERS
# ============================================================

def find_column(df, names):

    lookup = {
        str(c).lower(): c
        for c in df.columns
    }

    for name in names:

        if name.lower() in lookup:
            return lookup[name.lower()]

    return None


def to_utc_series(values):

    return pd.to_datetime(
        values,
        utc=True,
        errors="coerce"
    )


# ============================================================
# 2. FIND HISTORICALLY VALID 2026 TRADE SOURCE
#
# Highest priority:
# HISTORICAL_REPLAY_LIVE_TRADES
#
# We will NOT silently accept the wrong dataset.
# ============================================================

TRADE_SOURCE_CANDIDATES = [
    "HISTORICAL_REPLAY_LIVE_TRADES",
    "FINAL_TOURNAMENT_TRADES",
    "CHAMPION_REINTEGRATION_TRADES",
    "CLEAN_BASE_TRADES",
]


source_diagnostics = []

EXPECTED_SOURCE_NAME = None
EXPECTED_SOURCE_RAW = None


for name in TRADE_SOURCE_CANDIDATES:

    obj = globals().get(name)

    if not isinstance(obj, pd.DataFrame):
        continue

    temp = obj.copy()

    # --------------------------------------------------------
    # Feature-set filter if available
    # --------------------------------------------------------

    feature_col = find_column(
        temp,
        [
            "feature_set",
            "strategy",
            "champion",
        ]
    )

    if feature_col is not None:

        values = (
            temp[feature_col]
            .astype(str)
            .str.upper()
        )

        if (
            values
            ==
            "BASE_PLUS_REGIME"
        ).any():

            temp = temp.loc[
                values == "BASE_PLUS_REGIME"
            ].copy()


    # --------------------------------------------------------
    # Year filter
    # --------------------------------------------------------

    year_col = find_column(
        temp,
        [
            "test_year",
            "year",
        ]
    )

    if year_col is not None:

        years = pd.to_numeric(
            temp[year_col],
            errors="coerce"
        )

        if (years == TEST_YEAR).any():

            temp = temp.loc[
                years == TEST_YEAR
            ].copy()


    # --------------------------------------------------------
    # Signal time
    # --------------------------------------------------------

    signal_col = find_column(
        temp,
        [
            "signal_time",
            "timestamp",
        ]
    )

    if signal_col is not None:

        signal_times = to_utc_series(
            temp[signal_col]
        )

    else:

        signal_times = to_utc_series(
            temp.index
        )


    valid_time = signal_times.notna()

    temp = temp.loc[
        valid_time
    ].copy()

    signal_times = signal_times[
        valid_time
    ]


    if len(temp):

        # If the dataframe was not already year-filtered,
        # enforce year from signal timestamp.

        year_mask = (
            signal_times.dt.year
            ==
            TEST_YEAR
        )

        if year_mask.any():

            temp = temp.loc[
                year_mask.values
            ].copy()

            signal_times = signal_times[
                year_mask
            ]


    source_diagnostics.append({
        "variable":
            name,
        "candidate_rows_2026":
            len(temp),
    })


    # We only accept the exact known 2026 historical replay size.

    if len(temp) == EXPECTED_TRADE_COUNT:

        EXPECTED_SOURCE_NAME = name
        EXPECTED_SOURCE_RAW = temp.copy()

        break


print("=" * 110)
print("2026 HISTORICAL SIGNAL SOURCE SEARCH")
print("=" * 110)

display(
    pd.DataFrame(source_diagnostics)
)


if EXPECTED_SOURCE_RAW is None:

    raise RuntimeError(
        "\n2026年のHistorically-valid 300 trade datasetを"
        "安全に特定できませんでした。\n\n"
        "HISTORICAL_REPLAY_LIVE_TRADESを作成した"
        "Historical Replayセルを先に実行してください。"
    )


print()
print(
    "Selected source:",
    EXPECTED_SOURCE_NAME
)

print(
    "Expected trades:",
    len(EXPECTED_SOURCE_RAW)
)


# ============================================================
# 3. NORMALIZE EXPECTED TRADE TABLE
# ============================================================

src = EXPECTED_SOURCE_RAW.copy()


# ------------------------------------------------------------
# Signal time
# ------------------------------------------------------------

signal_col = find_column(
    src,
    [
        "signal_time",
        "timestamp",
    ]
)

if signal_col is not None:

    signal_time = to_utc_series(
        src[signal_col]
    )

else:

    signal_time = to_utc_series(
        src.index
    )


if signal_time.isna().any():

    raise RuntimeError(
        "Expected trade source contains invalid signal timestamps."
    )


# ------------------------------------------------------------
# Side
# ------------------------------------------------------------

side_col = find_column(
    src,
    [
        "side",
        "audit_side",
        "action",
        "trade_side",
    ]
)

if side_col is None:

    raise RuntimeError(
        "Expected trade sourceにBUY/SELL列がありません。"
    )


side = (
    src[side_col]
    .astype(str)
    .str.upper()
    .reset_index(drop=True)
)


if not side.isin(
    ["BUY", "SELL"]
).all():

    bad = side.loc[
        ~side.isin(["BUY", "SELL"])
    ].unique()

    raise RuntimeError(
        f"Invalid trade sides found: {bad}"
    )


# ------------------------------------------------------------
# Position size
# ------------------------------------------------------------

size_col = find_column(
    src,
    [
        "position_size",
        "size",
        "sizing_scale",
    ]
)

if size_col is None:

    raise RuntimeError(
        "Expected historical tradesにposition_sizeがありません。"
    )


position_size = pd.to_numeric(
    src[size_col],
    errors="coerce"
).reset_index(drop=True)


if (
    position_size.isna().any()
    or
    (position_size <= 0).any()
):

    raise RuntimeError(
        "Invalid position_size in expected trades."
    )


# ------------------------------------------------------------
# Optional fields
# ------------------------------------------------------------

confidence_col = find_column(
    src,
    [
        "confidence",
    ]
)

pup_col = find_column(
    src,
    [
        "p_up",
        "calibrated_p_up",
    ]
)


# ------------------------------------------------------------
# Entry / exit times
#
# Prefer actual expected columns if they exist.
# Otherwise derive from frozen timing contract.
#
# Signal bar timestamp = bar OPEN time
# Signal closes +15m
# Entry = signal +15m
# Exit  = entry +30m
# ------------------------------------------------------------

entry_time_col = find_column(
    src,
    [
        "entry_time",
        "actual_entry_time",
        "planned_entry_time",
        "audit_entry_time",
    ]
)

exit_time_col = find_column(
    src,
    [
        "exit_time",
        "actual_exit_time",
        "planned_exit_time",
        "audit_exit_time",
    ]
)


signal_time = signal_time.reset_index(
    drop=True
)


derived_entry_time = (
    signal_time
    +
    pd.Timedelta(minutes=15)
)


derived_exit_time = (
    derived_entry_time
    +
    pd.Timedelta(minutes=HOLD_MINUTES)
)


if entry_time_col is not None:

    expected_entry_time = to_utc_series(
        src[entry_time_col]
    ).reset_index(drop=True)

else:

    expected_entry_time = (
        derived_entry_time.copy()
    )


if exit_time_col is not None:

    expected_exit_time = to_utc_series(
        src[exit_time_col]
    ).reset_index(drop=True)

else:

    expected_exit_time = (
        derived_exit_time.copy()
    )


if (
    expected_entry_time.isna().any()
    or
    expected_exit_time.isna().any()
):

    raise RuntimeError(
        "Invalid expected entry/exit timestamps."
    )


# ============================================================
# 4. BUILD NORMALIZED EXPECTED TABLE
# ============================================================

EXPECTED_2026_TRADES = pd.DataFrame({

    "signal_time":
        signal_time,

    "side":
        side,

    "position_size":
        position_size,

    "entry_time":
        expected_entry_time,

    "exit_time":
        expected_exit_time,
})


if confidence_col is not None:

    EXPECTED_2026_TRADES[
        "confidence"
    ] = pd.to_numeric(
        src[confidence_col],
        errors="coerce"
    ).reset_index(drop=True)


if pup_col is not None:

    EXPECTED_2026_TRADES[
        "p_up"
    ] = pd.to_numeric(
        src[pup_col],
        errors="coerce"
    ).reset_index(drop=True)


EXPECTED_2026_TRADES = (
    EXPECTED_2026_TRADES
    .sort_values("signal_time")
    .reset_index(drop=True)
)


# ============================================================
# 5. EXPECTED OPTIONAL REFERENCE COLUMNS
# ============================================================

expected_entry_price_col = find_column(
    src,
    [
        "entry_price",
        "audit_entry_price",
    ]
)

expected_exit_price_col = find_column(
    src,
    [
        "exit_price",
        "audit_exit_price",
    ]
)

expected_gross_col = find_column(
    src,
    [
        "gross_return",
    ]
)

expected_net_col = find_column(
    src,
    [
        "net_return",
    ]
)


# Re-order src identically by signal time for optional comparisons.

src_ordered = src.copy()

if signal_col is not None:

    src_ordered["_signal_time_norm"] = (
        to_utc_series(
            src_ordered[signal_col]
        ).values
    )

else:

    src_ordered["_signal_time_norm"] = (
        to_utc_series(
            src_ordered.index
        ).values
    )


src_ordered = (
    src_ordered
    .sort_values("_signal_time_norm")
    .reset_index(drop=True)
)


# ============================================================
# 6. FIND CANONICAL PRICE BARS
# ============================================================

BAR_SOURCE_NAME = None
BARS = None


for name in [
    "REPLAY_BARS",
    "PRODUCTION_CANONICAL_HISTORY",
    "bars",
]:

    obj = globals().get(name)

    if not isinstance(obj, pd.DataFrame):
        continue

    temp = obj.copy()

    # Prefer existing Production normalizer if available.

    if "normalize_live_bars" in globals():

        try:

            temp = normalize_live_bars(
                temp
            )

        except Exception:

            continue

    else:

        # Minimal safe normalization.

        cols = {
            str(c).lower(): c
            for c in temp.columns
        }

        needed = [
            "open",
            "high",
            "low",
            "close",
        ]

        if not all(
            c in cols
            for c in needed
        ):

            continue

        temp = temp[
            [
                cols["open"],
                cols["high"],
                cols["low"],
                cols["close"],
            ]
        ].copy()

        temp.columns = needed

        temp.index = pd.to_datetime(
            temp.index,
            utc=True,
            errors="coerce"
        )

        temp = (
            temp.loc[
                ~temp.index.isna()
            ]
            .sort_index()
        )


    # Must cover full 2026 replay.

    if (
        temp.index.min()
        <=
        EXPECTED_2026_TRADES[
            "entry_time"
        ].min()

        and

        temp.index.max()
        >=
        EXPECTED_2026_TRADES[
            "exit_time"
        ].max()
    ):

        BAR_SOURCE_NAME = name
        BARS = temp.copy()

        break


if BARS is None:

    raise RuntimeError(
        "2026 Execution Replayに必要なCanonical price barsを"
        "安全に特定できませんでした。"
    )


print()
print("=" * 110)
print("CANONICAL PRICE SOURCE")
print("=" * 110)

print(
    "Source:",
    BAR_SOURCE_NAME
)

print(
    "Rows:",
    f"{len(BARS):,}"
)

print(
    "Period:",
    BARS.index.min(),
    "->",
    BARS.index.max()
)


# ============================================================
# 7. STRICT TIMING CHECK
# ============================================================

timing_entry_diff = (
    EXPECTED_2026_TRADES[
        "entry_time"
    ]
    !=
    (
        EXPECTED_2026_TRADES[
            "signal_time"
        ]
        +
        pd.Timedelta(minutes=15)
    )
)


timing_exit_diff = (
    EXPECTED_2026_TRADES[
        "exit_time"
    ]
    !=
    (
        EXPECTED_2026_TRADES[
            "entry_time"
        ]
        +
        pd.Timedelta(minutes=30)
    )
)


ENTRY_TIMING_MISMATCH = int(
    timing_entry_diff.sum()
)

EXIT_TIMING_MISMATCH = int(
    timing_exit_diff.sum()
)


# ============================================================
# 8. DUPLICATE / OVERLAP CHECK
# ============================================================

duplicate_signals = int(

    EXPECTED_2026_TRADES[
        "signal_time"
    ].duplicated().sum()
)


duplicate_entries = int(

    EXPECTED_2026_TRADES[
        "entry_time"
    ].duplicated().sum()
)


overlap_count = 0

previous_exit = None


for _, row in EXPECTED_2026_TRADES.iterrows():

    entry = row[
        "entry_time"
    ]

    exit_ = row[
        "exit_time"
    ]


    # Same-time:
    # previous exit == next entry
    # is allowed because EXIT happens before ENTRY.

    if (
        previous_exit is not None
        and
        entry < previous_exit
    ):

        overlap_count += 1


    previous_exit = exit_


# ============================================================
# 9. CHECK ALL ENTRY / EXIT PRICE BARS EXIST
# ============================================================

missing_entry_times = [

    ts
    for ts in EXPECTED_2026_TRADES[
        "entry_time"
    ]

    if ts not in BARS.index
]


missing_exit_times = [

    ts
    for ts in EXPECTED_2026_TRADES[
        "exit_time"
    ]

    if ts not in BARS.index
]


if missing_entry_times or missing_exit_times:

    print()
    print("Missing entry times:")
    print(missing_entry_times[:20])

    print()
    print("Missing exit times:")
    print(missing_exit_times[:20])

    raise RuntimeError(
        "Canonical bar lookup failed for one or more "
        "historical executions."
    )


# ============================================================
# 10. HISTORICAL EXECUTION REPLAY
#
# This does NOT call the Production model.
#
# The signal decisions are already historically validated.
# We are testing execution/accounting only.
# ============================================================

STARTING_EQUITY = 1_000_000.0

equity = STARTING_EQUITY

replay_rows = []


for i, trade in EXPECTED_2026_TRADES.iterrows():

    signal_time = trade[
        "signal_time"
    ]

    entry_time = trade[
        "entry_time"
    ]

    exit_time = trade[
        "exit_time"
    ]

    side = trade[
        "side"
    ]

    size = float(
        trade[
            "position_size"
        ]
    )


    # --------------------------------------------------------
    # Exact executable boundary OPEN prices
    # --------------------------------------------------------

    entry_price = float(

        BARS.loc[
            entry_time,
            "open"
        ]
    )


    exit_price = float(

        BARS.loc[
            exit_time,
            "open"
        ]
    )


    # --------------------------------------------------------
    # Same accounting function as PaperExecutionEngine
    # --------------------------------------------------------

    returns = paper_trade_return(

        side=
            side,

        entry_price=
            entry_price,

        exit_price=
            exit_price,

        position_size=
            size,

        cost=
            BASE_COST
    )


    equity_before = float(
        equity
    )


    equity = (

        equity

        *
        (
            1.0

            +
            returns[
                "net_return"
            ]
        )
    )


    equity_after = float(
        equity
    )


    replay_rows.append({

        "trade_id":
            i + 1,

        "signal_time":
            signal_time,

        "side":
            side,

        "position_size":
            size,

        "entry_time":
            entry_time,

        "entry_price":
            entry_price,

        "exit_time":
            exit_time,

        "exit_price":
            exit_price,

        "gross_return":
            float(
                returns[
                    "gross_return"
                ]
            ),

        "cost":
            BASE_COST,

        "net_return":
            float(
                returns[
                    "net_return"
                ]
            ),

        "equity_before":
            equity_before,

        "equity_after":
            equity_after,
    })


PAPER_EXECUTION_REPLAY_2026 = pd.DataFrame(
    replay_rows
)


# ============================================================
# 11. COMPARE EXPECTED PRICES / RETURNS WHEN AVAILABLE
# ============================================================

comparison = pd.DataFrame({

    "trade_id":
        PAPER_EXECUTION_REPLAY_2026[
            "trade_id"
        ],

    "signal_time":
        PAPER_EXECUTION_REPLAY_2026[
            "signal_time"
        ],
})


# ------------------------------------------------------------
# Entry price
# ------------------------------------------------------------

if expected_entry_price_col is not None:

    expected_entry_prices = pd.to_numeric(
        src_ordered[
            expected_entry_price_col
        ],
        errors="coerce"
    )

    comparison[
        "entry_price_diff"
    ] = (

        PAPER_EXECUTION_REPLAY_2026[
            "entry_price"
        ].values

        -

        expected_entry_prices.values
    )


# ------------------------------------------------------------
# Exit price
# ------------------------------------------------------------

if expected_exit_price_col is not None:

    expected_exit_prices = pd.to_numeric(
        src_ordered[
            expected_exit_price_col
        ],
        errors="coerce"
    )

    comparison[
        "exit_price_diff"
    ] = (

        PAPER_EXECUTION_REPLAY_2026[
            "exit_price"
        ].values

        -

        expected_exit_prices.values
    )


# ------------------------------------------------------------
# Gross
# ------------------------------------------------------------

if expected_gross_col is not None:

    expected_gross = pd.to_numeric(
        src_ordered[
            expected_gross_col
        ],
        errors="coerce"
    )

    comparison[
        "gross_return_diff"
    ] = (

        PAPER_EXECUTION_REPLAY_2026[
            "gross_return"
        ].values

        -

        expected_gross.values
    )


# ------------------------------------------------------------
# Net
# ------------------------------------------------------------

if expected_net_col is not None:

    expected_net = pd.to_numeric(
        src_ordered[
            expected_net_col
        ],
        errors="coerce"
    )

    comparison[
        "net_return_diff"
    ] = (

        PAPER_EXECUTION_REPLAY_2026[
            "net_return"
        ].values

        -

        expected_net.values
    )


# ============================================================
# 12. MAX DIFFS
# ============================================================

def safe_max_abs(df, col):

    if col not in df.columns:
        return np.nan

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    ).abs()

    if values.notna().sum() == 0:
        return np.nan

    return float(
        values.max()
    )


max_entry_price_diff = safe_max_abs(
    comparison,
    "entry_price_diff"
)

max_exit_price_diff = safe_max_abs(
    comparison,
    "exit_price_diff"
)

max_gross_diff = safe_max_abs(
    comparison,
    "gross_return_diff"
)

max_net_diff = safe_max_abs(
    comparison,
    "net_return_diff"
)


# ============================================================
# 13. PERFORMANCE FUNCTIONS
# ============================================================

def performance_from_returns(
    returns
):

    r = pd.Series(
        returns,
        dtype=float
    ).dropna()


    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": np.nan,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }


    positive = r[
        r > 0
    ].sum()


    negative = -r[
        r < 0
    ].sum()


    if negative > 0:

        pf = (
            positive
            /
            negative
        )

    elif positive > 0:

        pf = np.inf

    else:

        pf = np.nan


    equity_curve = np.concatenate(
        [
            [1.0],
            np.cumprod(
                1.0
                +
                r.values
            ),
        ]
    )


    running_peak = np.maximum.accumulate(
        equity_curve
    )


    drawdown = (
        equity_curve
        /
        running_peak
        -
        1.0
    )


    growth = float(
        equity_curve[-1]
        -
        1.0
    )


    max_dd = float(
        drawdown.min()
    )


    if max_dd < 0:

        return_to_dd = (
            growth
            /
            abs(max_dd)
        )

    else:

        return_to_dd = np.inf


    return {

        "trades":
            int(
                len(r)
            ),

        "win_rate":
            float(
                (r > 0).mean()
            ),

        "avg_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(
                return_to_dd
            ),
    }


REPLAY_PERFORMANCE = performance_from_returns(

    PAPER_EXECUTION_REPLAY_2026[
        "net_return"
    ]
)


# ============================================================
# 14. EXPECTED PERFORMANCE
#
# Prefer original historical expected net returns.
# ============================================================

if expected_net_col is not None:

    EXPECTED_PERFORMANCE = performance_from_returns(

        pd.to_numeric(
            src_ordered[
                expected_net_col
            ],
            errors="coerce"
        )
    )

else:

    # Exact regression values previously produced by the
    # historically-valid 2026 Frozen Champion replay.

    EXPECTED_PERFORMANCE = {

        "trades":
            300,

        "win_rate":
            np.nan,

        "avg_return":
            0.00011777082708124947,

        "profit_factor":
            1.825282251120899,

        "growth":
            0.03588118888467018,

        "max_dd":
            -0.0051943106534220185,

        "return_to_dd":
            np.nan,
    }


# ============================================================
# 15. PERFORMANCE DIFF
# ============================================================

PERFORMANCE_COMPARISON = pd.DataFrame({

    "metric": [
        "trades",
        "avg_return",
        "profit_factor",
        "growth",
        "max_dd",
    ],

    "expected": [
        EXPECTED_PERFORMANCE[
            "trades"
        ],

        EXPECTED_PERFORMANCE[
            "avg_return"
        ],

        EXPECTED_PERFORMANCE[
            "profit_factor"
        ],

        EXPECTED_PERFORMANCE[
            "growth"
        ],

        EXPECTED_PERFORMANCE[
            "max_dd"
        ],
    ],

    "replay": [
        REPLAY_PERFORMANCE[
            "trades"
        ],

        REPLAY_PERFORMANCE[
            "avg_return"
        ],

        REPLAY_PERFORMANCE[
            "profit_factor"
        ],

        REPLAY_PERFORMANCE[
            "growth"
        ],

        REPLAY_PERFORMANCE[
            "max_dd"
        ],
    ],
})


PERFORMANCE_COMPARISON[
    "diff"
] = (

    PERFORMANCE_COMPARISON[
        "replay"
    ]

    -

    PERFORMANCE_COMPARISON[
        "expected"
    ]
)


# ============================================================
# 16. TRADE-LEVEL CHECKS
# ============================================================

price_entry_ok = (

    True

    if np.isnan(
        max_entry_price_diff
    )

    else

    max_entry_price_diff
    <=
    PRICE_TOL
)


price_exit_ok = (

    True

    if np.isnan(
        max_exit_price_diff
    )

    else

    max_exit_price_diff
    <=
    PRICE_TOL
)


gross_ok = (

    True

    if np.isnan(
        max_gross_diff
    )

    else

    max_gross_diff
    <=
    RETURN_TOL
)


net_ok = (

    True

    if np.isnan(
        max_net_diff
    )

    else

    max_net_diff
    <=
    RETURN_TOL
)


# ============================================================
# 17. PERFORMANCE CHECKS
# ============================================================

avg_ok = np.isclose(

    REPLAY_PERFORMANCE[
        "avg_return"
    ],

    EXPECTED_PERFORMANCE[
        "avg_return"
    ],

    atol=1e-12,
    rtol=0.0
)


pf_ok = np.isclose(

    REPLAY_PERFORMANCE[
        "profit_factor"
    ],

    EXPECTED_PERFORMANCE[
        "profit_factor"
    ],

    atol=1e-10,
    rtol=0.0
)


growth_ok = np.isclose(

    REPLAY_PERFORMANCE[
        "growth"
    ],

    EXPECTED_PERFORMANCE[
        "growth"
    ],

    atol=1e-10,
    rtol=0.0
)


dd_ok = np.isclose(

    REPLAY_PERFORMANCE[
        "max_dd"
    ],

    EXPECTED_PERFORMANCE[
        "max_dd"
    ],

    atol=1e-10,
    rtol=0.0
)


# ============================================================
# 18. FINAL EXECUTION CHECKS
# ============================================================

EXECUTION_REPLAY_CHECKS = {

    "expected_trade_count_300":

        len(
            EXPECTED_2026_TRADES
        )
        ==
        300,

    "executed_trade_count_300":

        len(
            PAPER_EXECUTION_REPLAY_2026
        )
        ==
        300,

    "missing_trades_zero":

        len(
            EXPECTED_2026_TRADES
        )
        ==
        len(
            PAPER_EXECUTION_REPLAY_2026
        ),

    "unexpected_trades_zero":

        len(
            PAPER_EXECUTION_REPLAY_2026
        )
        ==
        len(
            EXPECTED_2026_TRADES
        ),

    "duplicate_signals_zero":

        duplicate_signals
        ==
        0,

    "duplicate_entries_zero":

        duplicate_entries
        ==
        0,

    "overlap_zero":

        overlap_count
        ==
        0,

    "entry_timing_exact":

        ENTRY_TIMING_MISMATCH
        ==
        0,

    "exit_timing_exact":

        EXIT_TIMING_MISMATCH
        ==
        0,

    "entry_price_parity":

        bool(
            price_entry_ok
        ),

    "exit_price_parity":

        bool(
            price_exit_ok
        ),

    "gross_return_parity":

        bool(
            gross_ok
        ),

    "net_return_parity":

        bool(
            net_ok
        ),

    "avg_return_parity":

        bool(
            avg_ok
        ),

    "profit_factor_parity":

        bool(
            pf_ok
        ),

    "growth_parity":

        bool(
            growth_ok
        ),

    "max_dd_parity":

        bool(
            dd_ok
        ),
}


EXECUTION_REPLAY_OK = all(

    EXECUTION_REPLAY_CHECKS.values()
)


# ============================================================
# 19. REPORT
# ============================================================

print()
print("=" * 110)
print("2026 PAPER EXECUTION REPLAY SUMMARY")
print("=" * 110)

print(
    "Historical signal source:",
    EXPECTED_SOURCE_NAME
)

print(
    "Price source:",
    BAR_SOURCE_NAME
)

print(
    "Expected trades:",
    len(
        EXPECTED_2026_TRADES
    )
)

print(
    "Executed trades:",
    len(
        PAPER_EXECUTION_REPLAY_2026
    )
)

print(
    "Duplicate signals:",
    duplicate_signals
)

print(
    "Duplicate entries:",
    duplicate_entries
)

print(
    "Overlapping trades:",
    overlap_count
)

print(
    "Entry timing mismatches:",
    ENTRY_TIMING_MISMATCH
)

print(
    "Exit timing mismatches:",
    EXIT_TIMING_MISMATCH
)


print()
print("=" * 110)
print("MAX TRADE-LEVEL DIFFERENCES")
print("=" * 110)

print(
    "Entry price:",
    max_entry_price_diff
)

print(
    "Exit price:",
    max_exit_price_diff
)

print(
    "Gross return:",
    max_gross_diff
)

print(
    "Net return:",
    max_net_diff
)


print()
print("=" * 110)
print("PERFORMANCE COMPARISON")
print("=" * 110)

display(
    PERFORMANCE_COMPARISON
)


print()
print("=" * 110)
print("REPLAY PERFORMANCE")
print("=" * 110)

for key, value in (
    REPLAY_PERFORMANCE.items()
):

    print(
        f"{key}: {value}"
    )


print()
print("=" * 110)
print("FINAL CHECKS")
print("=" * 110)

for key, value in (
    EXECUTION_REPLAY_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


# ============================================================
# 20. SAVE OUTPUT
# ============================================================

REPLAY_OUTPUT_DIR = (

    Path(
        LIVE_CHAMPION_DIR
    )

    /

    "runtime"

    /

    "paper"

    /

    "historical_execution_replay_2026"
)


REPLAY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


trade_log_path = (

    REPLAY_OUTPUT_DIR
    /
    "paper_execution_replay_2026_trades.csv"
)


comparison_path = (

    REPLAY_OUTPUT_DIR
    /
    "paper_execution_replay_2026_comparison.csv"
)


performance_path = (

    REPLAY_OUTPUT_DIR
    /
    "paper_execution_replay_2026_performance.csv"
)


decision_path = (

    REPLAY_OUTPUT_DIR
    /
    "paper_execution_replay_2026_decision.json"
)


PAPER_EXECUTION_REPLAY_2026.to_csv(
    trade_log_path,
    index=False
)


comparison.to_csv(
    comparison_path,
    index=False
)


PERFORMANCE_COMPARISON.to_csv(
    performance_path,
    index=False
)


EXECUTION_REPLAY_DECISION = (

    "PASS_READY_FOR_STATE_RECOVERY_AND_SAFETY"

    if EXECUTION_REPLAY_OK

    else

    "FAIL_DO_NOT_START_FORWARD_PAPER_TRADING"
)


decision_payload = {

    "test":
        "2026_HISTORICAL_PAPER_EXECUTION_REPLAY",

    "champion":
        PRODUCTION_PAPER_ENGINE_CONTRACT[
            "champion_name"
        ],

    "historical_signal_source":
        EXPECTED_SOURCE_NAME,

    "price_source":
        BAR_SOURCE_NAME,

    "expected_trades":
        int(
            len(
                EXPECTED_2026_TRADES
            )
        ),

    "executed_trades":
        int(
            len(
                PAPER_EXECUTION_REPLAY_2026
            )
        ),

    "checks":
        {
            key:
                bool(value)

            for key, value
            in EXECUTION_REPLAY_CHECKS.items()
        },

    "performance_expected":
        {
            key:
                (
                    None
                    if not np.isfinite(value)
                    else float(value)
                )

            for key, value
            in EXPECTED_PERFORMANCE.items()
        },

    "performance_replay":
        {
            key:
                (
                    None
                    if not np.isfinite(value)
                    else float(value)
                )

            for key, value
            in REPLAY_PERFORMANCE.items()
        },

    "decision":
        EXECUTION_REPLAY_DECISION,
}


with open(
    decision_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        decision_payload,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 21. SAVE NOTEBOOK VARIABLES
# ============================================================

PRODUCTION_EXECUTION_REPLAY_2026 = (
    PAPER_EXECUTION_REPLAY_2026.copy()
)

PRODUCTION_EXECUTION_REPLAY_COMPARISON = (
    comparison.copy()
)

PRODUCTION_EXECUTION_REPLAY_PERFORMANCE = (
    REPLAY_PERFORMANCE.copy()
)

PRODUCTION_EXECUTION_REPLAY_CHECKS = (
    EXECUTION_REPLAY_CHECKS.copy()
)

PRODUCTION_EXECUTION_REPLAY_OK = bool(
    EXECUTION_REPLAY_OK
)

PRODUCTION_EXECUTION_REPLAY_DECISION = (
    EXECUTION_REPLAY_DECISION
)

PRODUCTION_EXECUTION_REPLAY_OUTPUT_DIR = str(
    REPLAY_OUTPUT_DIR
)


# ============================================================
# 22. FINAL DECISION
# ============================================================

print()
print("=" * 110)
print("FINAL DECISION")
print("=" * 110)

print(
    "2026 PAPER EXECUTION REPLAY PASSED:",
    EXECUTION_REPLAY_OK
)

print()

print(
    "FINAL DECISION:",
    EXECUTION_REPLAY_DECISION
)

print()

print(
    "Saved folder:",
    REPLAY_OUTPUT_DIR
)


if EXECUTION_REPLAY_OK:

    print()
    print(
        "Historical prediction parity: PASS"
    )

    print(
        "Historical execution parity: PASS"
    )

    print(
        "30-minute execution accounting: PASS"
    )

    print(
        "Cost / sizing / equity accounting: PASS"
    )

    print()

    print(
        "NEXT STEP:"
    )

    print(
        "STATE PERSISTENCE + RESTART RECOVERY + KILL SWITCH TEST."
    )

else:

    print()
    print(
        "STOP."
    )

    print(
        "Do NOT start Forward Paper Trading."
    )

    print(
        "Inspect the failed check(s) above before continuing."
    )


## 元セルindex 76
構文状態：valid


In [ ]:
# ============================================================
# 2026 HISTORICAL PAPER EXECUTION REPLAY
# ONE-CELL ROBUST VERSION v2
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Frozen Champion:
#   BASE_PLUS_REGIME
#
# Historically-valid 2026 trade signalsを使い、
# Paper Execution Layerを厳密に再現する。
#
# This cell:
#   - fixes DatetimeIndex / .dt error
#   - auto-detects the validated 300-trade source
#   - checks entry/exit timing
#   - checks duplicate / overlap
#   - verifies execution-price convention
#   - replays entry -> hold -> exit
#   - reproduces cost / sizing / PnL / equity
#   - compares PF / Growth / DD
#
# IMPORTANT
# ------------------------------------------------------------
# NO TRAINING
# NO OPTIMIZATION
# NO FEATURE CHANGE
# NO MODEL CHANGE
# NO REAL ORDERS
# ============================================================


from pathlib import Path
import json
import math

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 320)
pd.set_option("display.max_rows", 100)


# ============================================================
# 0. CONSTANTS
# ============================================================

TEST_YEAR = 2026

EXPECTED_TRADE_COUNT = 300

EXPECTED_CHAMPION = "BASE_PLUS_REGIME"

TIMEFRAME_MINUTES = 15

HOLD_MINUTES_EXPECTED = 30

PRICE_TOL = 1e-8

RETURN_TOL = 1e-10

SIZE_TOL = 1e-10

PERFORMANCE_TOL = 1e-9


# ============================================================
# 1. REQUIRED PREVIOUS STATE
# ============================================================

REQUIRED_OBJECTS = [

    "PRODUCTION_PAPER_ENGINE_BUILD_OK",

    "PRODUCTION_PAPER_ENGINE_CONTRACT",

    "PRODUCTION_CANONICAL_HISTORY",

    "LIVE_CHAMPION_DIR",

    "paper_trade_return",
]


missing_objects = [

    name
    for name in REQUIRED_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Execution Replayに必要なNotebook状態が不足しています。\n\n"

        f"Missing:\n{missing_objects}\n\n"

        "Paper Execution Engine Buildセルまで"
        "先に実行してください。"
    )


if not bool(
    PRODUCTION_PAPER_ENGINE_BUILD_OK
):

    raise RuntimeError(

        "Paper Execution Engine BuildがPASSしていません。\n"

        "Execution Replayへ進めません。"
    )


PAPER_CONTRACT = (
    PRODUCTION_PAPER_ENGINE_CONTRACT
)


if (
    str(
        PAPER_CONTRACT.get(
            "champion_name"
        )
    )
    !=
    EXPECTED_CHAMPION
):

    raise RuntimeError(

        "Frozen Champion mismatch.\n\n"

        f"Expected: {EXPECTED_CHAMPION}\n"

        f"Actual: "
        f"{PAPER_CONTRACT.get('champion_name')}"
    )


BASE_COST = float(

    PAPER_CONTRACT.get(
        "base_cost_return",
        4e-05
    )
)


HOLD_MINUTES = int(

    PAPER_CONTRACT.get(
        "holding_period_minutes",
        30
    )
)


if HOLD_MINUTES != HOLD_MINUTES_EXPECTED:

    raise RuntimeError(

        "Frozen 30-minute exit policy mismatch.\n"

        f"Actual holding period: {HOLD_MINUTES}"
    )


print("=" * 110)
print("2026 HISTORICAL PAPER EXECUTION REPLAY")
print("=" * 110)

print(
    "Champion:",
    EXPECTED_CHAMPION
)

print(
    "Year:",
    TEST_YEAR
)

print(
    "Expected trades:",
    EXPECTED_TRADE_COUNT
)

print(
    "Base cost:",
    BASE_COST
)

print(
    "Holding period:",
    HOLD_MINUTES,
    "minutes"
)


# ============================================================
# 2. GENERAL HELPERS
# ============================================================

def find_column(
    df,
    candidates
):

    lookup = {

        str(c)
        .strip()
        .lower():
            c

        for c in df.columns
    }


    for candidate in candidates:

        key = (

            str(candidate)
            .strip()
            .lower()
        )

        if key in lookup:

            return lookup[key]


    return None


def to_utc_series(
    values
):
    """
    IMPORTANT:
    Always returns a pandas Series.

    This permanently avoids:

    AttributeError:
    'DatetimeIndex' object has no attribute 'dt'
    """

    if isinstance(
        values,
        pd.Series
    ):

        s = (
            values
            .copy()
            .reset_index(
                drop=True
            )
        )


    elif isinstance(
        values,
        pd.Index
    ):

        s = pd.Series(
            values.to_numpy()
        )


    elif isinstance(
        values,
        np.ndarray
    ):

        s = pd.Series(
            values
        )


    elif isinstance(
        values,
        (list, tuple)
    ):

        s = pd.Series(
            list(
                values
            )
        )


    else:

        s = pd.Series(
            [values]
        )


    s = pd.to_datetime(

        s,

        utc=True,

        errors="coerce"
    )


    return (
        s
        .reset_index(
            drop=True
        )
    )


def safe_float(
    value,
    default=np.nan
):

    try:

        x = float(
            value
        )

        return x

    except Exception:

        return float(
            default
        )


def max_abs_or_nan(
    values
):

    s = pd.to_numeric(

        pd.Series(
            values
        ),

        errors="coerce"
    )


    s = (
        s
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )


    if len(s) == 0:

        return np.nan


    return float(
        s.abs().max()
    )


# ============================================================
# 3. NORMALIZE HISTORICAL TRADE SOURCE
# ============================================================

def normalize_candidate_trade_source(
    dataframe
):

    df = dataframe.copy()


    # --------------------------------------------------------
    # Champion filter
    # --------------------------------------------------------

    feature_col = find_column(

        df,

        [
            "feature_set",
            "strategy",
            "champion",
            "model_name",
        ]
    )


    if feature_col is not None:

        feature_values = (

            df[
                feature_col
            ]

            .astype(str)

            .str.strip()

            .str.upper()
        )


        exact_champion = (

            feature_values
            ==
            EXPECTED_CHAMPION
        )


        if exact_champion.any():

            df = (
                df.loc[
                    exact_champion
                ]
                .copy()
            )


    # --------------------------------------------------------
    # Explicit year filter
    # --------------------------------------------------------

    year_col = find_column(

        df,

        [
            "test_year",
            "year",
        ]
    )


    if year_col is not None:

        years = pd.to_numeric(

            df[
                year_col
            ],

            errors="coerce"
        )


        year_mask = (

            years
            ==
            TEST_YEAR
        )


        if year_mask.any():

            df = (
                df.loc[
                    year_mask
                ]
                .copy()
            )


    # --------------------------------------------------------
    # Signal timestamp
    # --------------------------------------------------------

    signal_col = find_column(

        df,

        [
            "signal_time",
            "timestamp",
            "signal_timestamp",
        ]
    )


    if signal_col is not None:

        signal_times = to_utc_series(

            df[
                signal_col
            ]
        )

    else:

        signal_times = to_utc_series(
            df.index
        )


    # --------------------------------------------------------
    # Invalid timestamp removal
    # POSITIONAL filtering intentionally used.
    # --------------------------------------------------------

    valid_mask = (

        signal_times
        .notna()
        .to_numpy()
    )


    df = (
        df.iloc[
            valid_mask
        ]
        .copy()
    )


    signal_times = (

        signal_times
        .iloc[
            valid_mask
        ]
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Actual timestamp year = 2026
    # --------------------------------------------------------

    if len(df) > 0:

        timestamp_year_mask = (

            signal_times
            .dt
            .year
            .eq(
                TEST_YEAR
            )
            .to_numpy()
        )


        df = (

            df.iloc[
                timestamp_year_mask
            ]
            .copy()
        )


        signal_times = (

            signal_times
            .iloc[
                timestamp_year_mask
            ]
            .reset_index(
                drop=True
            )
        )


    # --------------------------------------------------------
    # BUY / SELL only if side exists
    # --------------------------------------------------------

    side_col = find_column(

        df,

        [
            "side",
            "audit_side",
            "trade_side",
            "action",
        ]
    )


    if (
        side_col is not None
        and
        len(df) > 0
    ):

        sides = (

            df[
                side_col
            ]

            .astype(str)

            .str.strip()

            .str.upper()

            .reset_index(
                drop=True
            )
        )


        trade_mask = (

            sides
            .isin(
                ["BUY", "SELL"]
            )
            .to_numpy()
        )


        if trade_mask.any():

            df = (

                df.iloc[
                    trade_mask
                ]
                .copy()
            )


            signal_times = (

                signal_times
                .iloc[
                    trade_mask
                ]
                .reset_index(
                    drop=True
                )
            )


    df = (
        df
        .reset_index(
            drop=False
        )
    )


    df[
        "_signal_time_normalized"
    ] = signal_times.to_numpy()


    df = (

        df
        .sort_values(
            "_signal_time_normalized"
        )

        .reset_index(
            drop=True
        )
    )


    return df


# ============================================================
# 4. AUTO-DETECT EXACT 300-TRADE SOURCE
# ============================================================

TRADE_SOURCE_CANDIDATES = [

    "HISTORICAL_REPLAY_LIVE_TRADES",

    "FINAL_TOURNAMENT_TRADES",

    "CHAMPION_REINTEGRATION_TRADES",

    "CLEAN_BASE_TRADES",
]


SOURCE_DIAGNOSTICS_ROWS = []

EXPECTED_SOURCE_NAME = None

EXPECTED_SOURCE = None


for variable_name in (
    TRADE_SOURCE_CANDIDATES
):

    obj = globals().get(
        variable_name
    )


    if not isinstance(
        obj,
        pd.DataFrame
    ):

        SOURCE_DIAGNOSTICS_ROWS.append({

            "variable":
                variable_name,

            "exists":
                False,

            "raw_rows":
                0,

            "2026_trade_rows":
                0,
        })

        continue


    try:

        candidate = (
            normalize_candidate_trade_source(
                obj
            )
        )


        candidate_rows = len(
            candidate
        )


    except Exception as exc:

        SOURCE_DIAGNOSTICS_ROWS.append({

            "variable":
                variable_name,

            "exists":
                True,

            "raw_rows":
                len(obj),

            "2026_trade_rows":
                -1,

            "error":
                f"{type(exc).__name__}: {exc}",
        })

        continue


    SOURCE_DIAGNOSTICS_ROWS.append({

        "variable":
            variable_name,

        "exists":
            True,

        "raw_rows":
            len(obj),

        "2026_trade_rows":
            candidate_rows,
    })


    if (

        EXPECTED_SOURCE is None

        and

        candidate_rows
        ==
        EXPECTED_TRADE_COUNT

    ):

        EXPECTED_SOURCE_NAME = (
            variable_name
        )

        EXPECTED_SOURCE = (
            candidate.copy()
        )


SOURCE_DIAGNOSTICS = pd.DataFrame(
    SOURCE_DIAGNOSTICS_ROWS
)


print()
print("=" * 110)
print("HISTORICAL SIGNAL SOURCE SEARCH")
print("=" * 110)

display(
    SOURCE_DIAGNOSTICS
)


if EXPECTED_SOURCE is None:

    print()
    print("Available candidate columns:")


    for variable_name in (
        TRADE_SOURCE_CANDIDATES
    ):

        obj = globals().get(
            variable_name
        )


        if isinstance(
            obj,
            pd.DataFrame
        ):

            print()
            print(
                variable_name
            )

            print(
                list(
                    obj.columns
                )
            )


    raise RuntimeError(

        "\nHistorically-validな2026年300取引を"
        "自動特定できませんでした。\n\n"

        "これはChampionのFAILではありません。\n"

        "Historical Replay trade sourceの"
        "Notebook変数を確認してください。"
    )


print()
print(
    "SELECTED SOURCE:",
    EXPECTED_SOURCE_NAME
)

print(
    "2026 Trades:",
    len(
        EXPECTED_SOURCE
    )
)

print(
    "SOURCE CHECK PASSED: True"
)


# ============================================================
# 5. NORMALIZE EXPECTED TRADES
# ============================================================

src = EXPECTED_SOURCE.copy()


signal_time = to_utc_series(

    src[
        "_signal_time_normalized"
    ]
)


# ------------------------------------------------------------
# Side
# ------------------------------------------------------------

side_col = find_column(

    src,

    [
        "side",
        "audit_side",
        "trade_side",
        "action",
    ]
)


if side_col is None:

    raise RuntimeError(

        "Historically-valid tradesに"
        "BUY/SELL side columnがありません。"
    )


side = (

    src[
        side_col
    ]

    .astype(str)

    .str.strip()

    .str.upper()

    .reset_index(
        drop=True
    )
)


if not side.isin(
    ["BUY", "SELL"]
).all():

    raise RuntimeError(

        "Historical trade sourceに"
        "BUY/SELL以外のtrade sideがあります。"
    )


# ------------------------------------------------------------
# Position size
# ------------------------------------------------------------

size_col = find_column(

    src,

    [
        "position_size",
        "size",
    ]
)


if size_col is None:

    raise RuntimeError(

        "Historical tradesにposition_size列がありません。"
    )


position_size = pd.to_numeric(

    src[
        size_col
    ],

    errors="coerce"
).reset_index(
    drop=True
)


if (

    position_size.isna().any()

    or

    (
        position_size
        <=
        0
    ).any()

):

    raise RuntimeError(

        "Historical tradesのposition_sizeに"
        "不正値があります。"
    )


# ------------------------------------------------------------
# Optional probability fields
# ------------------------------------------------------------

confidence_col = find_column(

    src,

    [
        "confidence",
    ]
)


p_up_col = find_column(

    src,

    [
        "p_up",
        "calibrated_p_up",
    ]
)


# ============================================================
# 6. ENTRY / EXIT TIMESTAMPS
# ============================================================

entry_time_col = find_column(

    src,

    [
        "entry_time",
        "actual_entry_time",
        "planned_entry_time",
        "audit_entry_time",
    ]
)


exit_time_col = find_column(

    src,

    [
        "exit_time",
        "actual_exit_time",
        "planned_exit_time",
        "audit_exit_time",
    ]
)


# Frozen live timing convention:
#
# signal bar index = BAR OPEN
#
# signal closes 15m later
# entry = signal + 15m
# exit  = entry + 30m

derived_entry_time = (

    signal_time

    +

    pd.Timedelta(
        minutes=15
    )
)


derived_exit_time = (

    derived_entry_time

    +

    pd.Timedelta(
        minutes=HOLD_MINUTES
    )
)


# ------------------------------------------------------------
# Use stored timestamps only when every value is valid.
# Otherwise use frozen timing rule.
# ------------------------------------------------------------

if entry_time_col is not None:

    stored_entry = to_utc_series(

        src[
            entry_time_col
        ]
    )


    if stored_entry.notna().all():

        entry_time = (
            stored_entry.copy()
        )

    else:

        entry_time = (
            derived_entry_time.copy()
        )

else:

    entry_time = (
        derived_entry_time.copy()
    )


if exit_time_col is not None:

    stored_exit = to_utc_series(

        src[
            exit_time_col
        ]
    )


    if stored_exit.notna().all():

        exit_time = (
            stored_exit.copy()
        )

    else:

        exit_time = (
            derived_exit_time.copy()
        )

else:

    exit_time = (
        derived_exit_time.copy()
    )


# ============================================================
# 7. EXPECTED TRADE TABLE
# ============================================================

EXPECTED_2026_TRADES = pd.DataFrame({

    "trade_id":
        np.arange(
            1,
            len(src) + 1
        ),

    "signal_time":
        signal_time,

    "side":
        side,

    "position_size":
        position_size,

    "entry_time":
        entry_time,

    "exit_time":
        exit_time,
})


if confidence_col is not None:

    EXPECTED_2026_TRADES[
        "confidence"
    ] = pd.to_numeric(

        src[
            confidence_col
        ],

        errors="coerce"
    ).reset_index(
        drop=True
    )


if p_up_col is not None:

    EXPECTED_2026_TRADES[
        "p_up"
    ] = pd.to_numeric(

        src[
            p_up_col
        ],

        errors="coerce"
    ).reset_index(
        drop=True
    )


EXPECTED_2026_TRADES = (

    EXPECTED_2026_TRADES

    .sort_values(
        "signal_time"
    )

    .reset_index(
        drop=True
    )
)


EXPECTED_2026_TRADES[
    "trade_id"
] = np.arange(
    1,
    len(
        EXPECTED_2026_TRADES
    ) + 1
)


# ============================================================
# 8. TIMING INTEGRITY
# ============================================================

ENTRY_TIMING_MISMATCHES = int(

    (
        EXPECTED_2026_TRADES[
            "entry_time"
        ]

        !=

        (
            EXPECTED_2026_TRADES[
                "signal_time"
            ]

            +

            pd.Timedelta(
                minutes=15
            )
        )
    )
    .sum()
)


EXIT_TIMING_MISMATCHES = int(

    (
        EXPECTED_2026_TRADES[
            "exit_time"
        ]

        !=

        (
            EXPECTED_2026_TRADES[
                "entry_time"
            ]

            +

            pd.Timedelta(
                minutes=30
            )
        )
    )
    .sum()
)


INVALID_HOLDING_PERIODS = int(

    (
        EXPECTED_2026_TRADES[
            "exit_time"
        ]

        <=

        EXPECTED_2026_TRADES[
            "entry_time"
        ]
    )
    .sum()
)


# ============================================================
# 9. DUPLICATE / OVERLAP
# ============================================================

DUPLICATE_SIGNALS = int(

    EXPECTED_2026_TRADES[
        "signal_time"
    ]
    .duplicated()
    .sum()
)


DUPLICATE_ENTRIES = int(

    EXPECTED_2026_TRADES[
        "entry_time"
    ]
    .duplicated()
    .sum()
)


OVERLAP_COUNT = 0

running_exit = None


for _, row in (

    EXPECTED_2026_TRADES

    .sort_values(
        "entry_time"
    )

    .iterrows()
):

    this_entry = row[
        "entry_time"
    ]

    this_exit = row[
        "exit_time"
    ]


    # Exact equality is allowed:
    #
    # old position exits at 15:45
    # new position enters at 15:45
    #
    # EXIT is processed first.

    if (

        running_exit is not None

        and

        this_entry
        <
        running_exit

    ):

        OVERLAP_COUNT += 1


    if (

        running_exit is None

        or

        this_exit
        >
        running_exit

    ):

        running_exit = (
            this_exit
        )


# ============================================================
# 10. CANONICAL PRICE DATA
# ============================================================

def normalize_price_bars(
    dataframe
):

    df = dataframe.copy()


    df.columns = [

        str(c)
        .strip()
        .lower()

        for c in df.columns
    ]


    # --------------------------------------------------------
    # Timestamp may be column or index
    # --------------------------------------------------------

    if not isinstance(
        df.index,
        pd.DatetimeIndex
    ):

        timestamp_col = find_column(

            df,

            [
                "timestamp",
                "datetime",
                "time",
            ]
        )


        if timestamp_col is None:

            raise RuntimeError(

                "Canonical barsにtimestampを"
                "特定できません。"
            )


        df.index = pd.to_datetime(

            df[
                timestamp_col
            ],

            utc=True,

            errors="coerce"
        )

    else:

        df.index = pd.to_datetime(

            df.index,

            utc=True,

            errors="coerce"
        )


    df = df.loc[
        ~df.index.isna()
    ].copy()


    required = [

        "open",
        "high",
        "low",
        "close",
    ]


    missing = [

        c
        for c in required

        if c not in df.columns
    ]


    if missing:

        raise RuntimeError(

            f"Canonical price columns missing: {missing}"
        )


    df = df[
        required
    ].copy()


    for c in required:

        df[c] = pd.to_numeric(

            df[c],

            errors="coerce"
        )


    df = df.dropna(
        subset=required
    )


    df = (
        df
        .sort_index()
    )


    if df.index.duplicated().any():

        duplicate_count = int(

            df.index
            .duplicated()
            .sum()
        )


        raise RuntimeError(

            "Canonical price barsに"
            f"duplicate timestampsがあります: {duplicate_count}"
        )


    return df


BARS = normalize_price_bars(

    PRODUCTION_CANONICAL_HISTORY
)


print()
print("=" * 110)
print("CANONICAL PRICE DATA")
print("=" * 110)

print(
    "Rows:",
    f"{len(BARS):,}"
)

print(
    "Period:",
    BARS.index.min(),
    "->",
    BARS.index.max()
)


# ============================================================
# 11. ORIGINAL EXPECTED EXECUTION COLUMNS
# ============================================================

expected_entry_price_col = find_column(

    src,

    [
        "entry_price",
        "audit_entry_price",
    ]
)


expected_exit_price_col = find_column(

    src,

    [
        "exit_price",
        "audit_exit_price",
    ]
)


expected_gross_col = find_column(

    src,

    [
        "gross_return",
    ]
)


expected_net_col = find_column(

    src,

    [
        "net_return",
    ]
)


# ============================================================
# 12. BOUNDARY PRICE FUNCTIONS
# ============================================================

def get_boundary_price(
    timestamp,
    convention
):

    ts = pd.Timestamp(
        timestamp
    )


    if ts.tzinfo is None:

        ts = ts.tz_localize(
            "UTC"
        )

    else:

        ts = ts.tz_convert(
            "UTC"
        )


    # --------------------------------------------------------
    # Convention A
    #
    # Price = OPEN of bar beginning at execution timestamp.
    # --------------------------------------------------------

    if convention == "OPEN_AT_BOUNDARY":

        if ts not in BARS.index:

            return np.nan


        return float(

            BARS.loc[
                ts,
                "open"
            ]
        )


    # --------------------------------------------------------
    # Convention B
    #
    # Execution at boundary = CLOSE of the bar that just closed.
    #
    # Example:
    # 10:00 bar closes 10:15
    # price at boundary 10:15 = close of 10:00 bar
    # --------------------------------------------------------

    elif convention == "PREVIOUS_BAR_CLOSE":

        previous_ts = (

            ts

            -

            pd.Timedelta(
                minutes=15
            )
        )


        if previous_ts not in BARS.index:

            return np.nan


        return float(

            BARS.loc[
                previous_ts,
                "close"
            ]
        )


    else:

        raise ValueError(

            f"Unknown price convention: {convention}"
        )


# ============================================================
# 13. HISTORICAL EXECUTION STATE MACHINE
# ============================================================

def run_execution_replay(
    convention
):

    # --------------------------------------------------------
    # Construct execution events
    #
    # EXIT priority = 0
    # ENTRY priority = 1
    #
    # This allows:
    # previous exit == next entry
    # --------------------------------------------------------

    events = []


    for _, trade in (
        EXPECTED_2026_TRADES.iterrows()
    ):

        trade_dict = (
            trade.to_dict()
        )


        events.append({

            "timestamp":
                trade[
                    "entry_time"
                ],

            "priority":
                1,

            "event":
                "ENTRY",

            "trade":
                trade_dict,
        })


        events.append({

            "timestamp":
                trade[
                    "exit_time"
                ],

            "priority":
                0,

            "event":
                "EXIT",

            "trade":
                trade_dict,
        })


    events = sorted(

        events,

        key=lambda x: (

            x[
                "timestamp"
            ],

            x[
                "priority"
            ]
        )
    )


    equity = (
        1_000_000.0
    )


    open_position = None


    trade_log = []


    errors = []


    entry_count = 0

    exit_count = 0


    for event in events:

        timestamp = event[
            "timestamp"
        ]

        trade = event[
            "trade"
        ]


        # ----------------------------------------------------
        # ENTRY
        # ----------------------------------------------------

        if (
            event[
                "event"
            ]
            ==
            "ENTRY"
        ):

            if open_position is not None:

                errors.append({

                    "timestamp":
                        timestamp,

                    "error":
                        "OVERLAP_ENTRY",

                    "trade_id":
                        trade[
                            "trade_id"
                        ],

                    "open_trade_id":
                        open_position[
                            "trade_id"
                        ],
                })

                continue


            entry_price = get_boundary_price(

                timestamp,

                convention
            )


            if not np.isfinite(
                entry_price
            ):

                errors.append({

                    "timestamp":
                        timestamp,

                    "error":
                        "MISSING_ENTRY_PRICE",

                    "trade_id":
                        trade[
                            "trade_id"
                        ],
                })

                continue


            open_position = {

                "trade_id":
                    int(
                        trade[
                            "trade_id"
                        ]
                    ),

                "signal_time":
                    trade[
                        "signal_time"
                    ],

                "side":
                    str(
                        trade[
                            "side"
                        ]
                    ),

                "position_size":
                    float(
                        trade[
                            "position_size"
                        ]
                    ),

                "entry_time":
                    timestamp,

                "entry_price":
                    float(
                        entry_price
                    ),

                "expected_exit_time":
                    trade[
                        "exit_time"
                    ],
            }


            entry_count += 1


        # ----------------------------------------------------
        # EXIT
        # ----------------------------------------------------

        elif (
            event[
                "event"
            ]
            ==
            "EXIT"
        ):

            if open_position is None:

                errors.append({

                    "timestamp":
                        timestamp,

                    "error":
                        "EXIT_WITHOUT_POSITION",

                    "trade_id":
                        trade[
                            "trade_id"
                        ],
                })

                continue


            if (

                int(
                    open_position[
                        "trade_id"
                    ]
                )

                !=

                int(
                    trade[
                        "trade_id"
                    ]
                )

            ):

                errors.append({

                    "timestamp":
                        timestamp,

                    "error":
                        "WRONG_POSITION_AT_EXIT",

                    "expected_trade_id":
                        trade[
                            "trade_id"
                        ],

                    "open_trade_id":
                        open_position[
                            "trade_id"
                        ],
                })

                continue


            exit_price = get_boundary_price(

                timestamp,

                convention
            )


            if not np.isfinite(
                exit_price
            ):

                errors.append({

                    "timestamp":
                        timestamp,

                    "error":
                        "MISSING_EXIT_PRICE",

                    "trade_id":
                        trade[
                            "trade_id"
                        ],
                })

                continue


            returns = paper_trade_return(

                side=
                    open_position[
                        "side"
                    ],

                entry_price=
                    open_position[
                        "entry_price"
                    ],

                exit_price=
                    exit_price,

                position_size=
                    open_position[
                        "position_size"
                    ],

                cost=
                    BASE_COST
            )


            equity_before = (
                float(
                    equity
                )
            )


            equity = (

                equity

                *

                (
                    1.0

                    +

                    returns[
                        "net_return"
                    ]
                )
            )


            trade_log.append({

                "trade_id":
                    open_position[
                        "trade_id"
                    ],

                "signal_time":
                    open_position[
                        "signal_time"
                    ],

                "side":
                    open_position[
                        "side"
                    ],

                "position_size":
                    open_position[
                        "position_size"
                    ],

                "entry_time":
                    open_position[
                        "entry_time"
                    ],

                "entry_price":
                    open_position[
                        "entry_price"
                    ],

                "exit_time":
                    timestamp,

                "exit_price":
                    float(
                        exit_price
                    ),

                "gross_return":
                    float(
                        returns[
                            "gross_return"
                        ]
                    ),

                "cost":
                    float(
                        BASE_COST
                    ),

                "net_return":
                    float(
                        returns[
                            "net_return"
                        ]
                    ),

                "equity_before":
                    equity_before,

                "equity_after":
                    float(
                        equity
                    ),
            })


            open_position = None

            exit_count += 1


    if open_position is not None:

        errors.append({

            "timestamp":
                open_position[
                    "expected_exit_time"
                ],

            "error":
                "POSITION_LEFT_OPEN",

            "trade_id":
                open_position[
                    "trade_id"
                ],
        })


    log_df = pd.DataFrame(
        trade_log
    )


    error_df = pd.DataFrame(
        errors
    )


    return {

        "convention":
            convention,

        "trade_log":
            log_df,

        "errors":
            error_df,

        "entry_count":
            int(
                entry_count
            ),

        "exit_count":
            int(
                exit_count
            ),

        "final_equity":
            float(
                equity
            ),

        "flat_at_end":
            open_position is None,
    }


# ============================================================
# 14. RUN BOTH POSSIBLE EXECUTION PRICE CONVENTIONS
# ============================================================

CANDIDATE_CONVENTIONS = [

    "PREVIOUS_BAR_CLOSE",

    "OPEN_AT_BOUNDARY",
]


REPLAY_CANDIDATES = {

    convention:
        run_execution_replay(
            convention
        )

    for convention in CANDIDATE_CONVENTIONS
}


# ============================================================
# 15. ALIGN ORIGINAL EXPECTED VALUES
# ============================================================

EXPECTED_REFERENCE = src.copy()


EXPECTED_REFERENCE = (

    EXPECTED_REFERENCE

    .sort_values(
        "_signal_time_normalized"
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 16. TRADE-LEVEL PARITY FOR EACH CONVENTION
# ============================================================

CONVENTION_DIAGNOSTICS_ROWS = []


def calculate_convention_diagnostics(
    result
):

    replay = (
        result[
            "trade_log"
        ]
        .copy()
    )


    diagnostics = {

        "convention":
            result[
                "convention"
            ],

        "entries":
            result[
                "entry_count"
            ],

        "exits":
            result[
                "exit_count"
            ],

        "trades":
            len(
                replay
            ),

        "engine_errors":
            len(
                result[
                    "errors"
                ]
            ),

        "flat_at_end":
            bool(
                result[
                    "flat_at_end"
                ]
            ),

        "max_entry_price_diff":
            np.nan,

        "max_exit_price_diff":
            np.nan,

        "max_gross_return_diff":
            np.nan,

        "max_net_return_diff":
            np.nan,
    }


    if len(replay) != len(
        EXPECTED_REFERENCE
    ):

        diagnostics[
            "trade_level_pass"
        ] = False

        return diagnostics


    # --------------------------------------------------------
    # Expected entry price
    # --------------------------------------------------------

    if expected_entry_price_col is not None:

        expected_values = pd.to_numeric(

            EXPECTED_REFERENCE[
                expected_entry_price_col
            ],

            errors="coerce"
        )


        diagnostics[
            "max_entry_price_diff"
        ] = max_abs_or_nan(

            replay[
                "entry_price"
            ].to_numpy()

            -

            expected_values.to_numpy()
        )


    # --------------------------------------------------------
    # Expected exit price
    # --------------------------------------------------------

    if expected_exit_price_col is not None:

        expected_values = pd.to_numeric(

            EXPECTED_REFERENCE[
                expected_exit_price_col
            ],

            errors="coerce"
        )


        diagnostics[
            "max_exit_price_diff"
        ] = max_abs_or_nan(

            replay[
                "exit_price"
            ].to_numpy()

            -

            expected_values.to_numpy()
        )


    # --------------------------------------------------------
    # Expected gross
    # --------------------------------------------------------

    if expected_gross_col is not None:

        expected_values = pd.to_numeric(

            EXPECTED_REFERENCE[
                expected_gross_col
            ],

            errors="coerce"
        )


        diagnostics[
            "max_gross_return_diff"
        ] = max_abs_or_nan(

            replay[
                "gross_return"
            ].to_numpy()

            -

            expected_values.to_numpy()
        )


    # --------------------------------------------------------
    # Expected net
    # --------------------------------------------------------

    if expected_net_col is not None:

        expected_values = pd.to_numeric(

            EXPECTED_REFERENCE[
                expected_net_col
            ],

            errors="coerce"
        )


        diagnostics[
            "max_net_return_diff"
        ] = max_abs_or_nan(

            replay[
                "net_return"
            ].to_numpy()

            -

            expected_values.to_numpy()
        )


    checks = []


    for key, tolerance in [

        (
            "max_entry_price_diff",
            PRICE_TOL
        ),

        (
            "max_exit_price_diff",
            PRICE_TOL
        ),

        (
            "max_gross_return_diff",
            RETURN_TOL
        ),

        (
            "max_net_return_diff",
            RETURN_TOL
        ),
    ]:

        value = diagnostics[
            key
        ]


        if np.isfinite(
            value
        ):

            checks.append(

                value
                <=
                tolerance
            )


    # Need:
    # no state-machine errors
    # 300 trades
    # flat at end
    # and every available historical parity field exact

    diagnostics[
        "trade_level_pass"
    ] = bool(

        result[
            "entry_count"
        ]
        ==
        EXPECTED_TRADE_COUNT

        and

        result[
            "exit_count"
        ]
        ==
        EXPECTED_TRADE_COUNT

        and

        len(
            replay
        )
        ==
        EXPECTED_TRADE_COUNT

        and

        len(
            result[
                "errors"
            ]
        )
        ==
        0

        and

        result[
            "flat_at_end"
        ]

        and

        (
            all(checks)

            if checks

            else True
        )
    )


    return diagnostics


for convention in (
    CANDIDATE_CONVENTIONS
):

    diagnostics = (

        calculate_convention_diagnostics(

            REPLAY_CANDIDATES[
                convention
            ]
        )
    )


    CONVENTION_DIAGNOSTICS_ROWS.append(
        diagnostics
    )


CONVENTION_DIAGNOSTICS = pd.DataFrame(

    CONVENTION_DIAGNOSTICS_ROWS
)


print()
print("=" * 110)
print("EXECUTION PRICE CONVENTION DIAGNOSTIC")
print("=" * 110)

display(
    CONVENTION_DIAGNOSTICS
)


# ============================================================
# 17. SELECT FROZEN EXECUTION CONVENTION
#
# This is NOT optimization.
#
# We are recovering the exact accounting semantics already
# used by the validated historical strategy.
# ============================================================

passing_conventions = (

    CONVENTION_DIAGNOSTICS.loc[

        CONVENTION_DIAGNOSTICS[
            "trade_level_pass"
        ]
        ==
        True,

        "convention"

    ]
    .tolist()
)


if len(
    passing_conventions
) == 0:

    # --------------------------------------------------------
    # If original trade dataframe did not store prices/returns,
    # PREVIOUS_BAR_CLOSE remains canonical boundary semantics.
    # Performance parity later will still have to pass.
    # --------------------------------------------------------

    no_reference_fields = all(

        c is None

        for c in [

            expected_entry_price_col,

            expected_exit_price_col,

            expected_gross_col,

            expected_net_col,
        ]
    )


    if no_reference_fields:

        SELECTED_PRICE_CONVENTION = (
            "PREVIOUS_BAR_CLOSE"
        )

    else:

        SELECTED_PRICE_CONVENTION = None


elif (
    "PREVIOUS_BAR_CLOSE"
    in
    passing_conventions
):

    # Deterministic preference:
    # closed 15m bar boundary price.

    SELECTED_PRICE_CONVENTION = (
        "PREVIOUS_BAR_CLOSE"
    )


else:

    SELECTED_PRICE_CONVENTION = (
        passing_conventions[0]
    )


if SELECTED_PRICE_CONVENTION is None:

    print()
    print("=" * 110)
    print("EXECUTION PRICE PARITY FAILED")
    print("=" * 110)

    print(
        "Neither execution-price convention reproduced "
        "the historical Frozen trades."
    )


    for convention in (
        CANDIDATE_CONVENTIONS
    ):

        errors = (

            REPLAY_CANDIDATES[
                convention
            ][
                "errors"
            ]
        )


        if len(errors):

            print()
            print(
                convention,
                "errors:"
            )

            display(
                errors.head(
                    20
                )
            )


    raise RuntimeError(

        "Execution price semantics could not be reproduced.\n"

        "STOP before Forward Paper Trading."
    )


print()
print(
    "SELECTED EXECUTION PRICE CONVENTION:",
    SELECTED_PRICE_CONVENTION
)


FINAL_REPLAY_RESULT = (

    REPLAY_CANDIDATES[
        SELECTED_PRICE_CONVENTION
    ]
)


PAPER_EXECUTION_REPLAY_2026 = (

    FINAL_REPLAY_RESULT[
        "trade_log"
    ]
    .copy()
)


PAPER_EXECUTION_REPLAY_ERRORS = (

    FINAL_REPLAY_RESULT[
        "errors"
    ]
    .copy()
)


# ============================================================
# 18. PERFORMANCE CALCULATION
# ============================================================

def performance_from_returns(
    returns
):

    r = pd.to_numeric(

        pd.Series(
            returns
        ),

        errors="coerce"
    ).dropna()


    if len(r) == 0:

        return {

            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "profit_factor":
                np.nan,

            "growth":
                np.nan,

            "max_dd":
                np.nan,

            "return_to_dd":
                np.nan,
        }


    gains = float(

        r.loc[
            r > 0
        ]
        .sum()
    )


    losses = float(

        -r.loc[
            r < 0
        ]
        .sum()
    )


    if losses > 0:

        pf = (
            gains
            /
            losses
        )

    elif gains > 0:

        pf = np.inf

    else:

        pf = np.nan


    equity_curve = np.concatenate(

        [
            [1.0],

            np.cumprod(
                1.0
                +
                r.to_numpy()
            ),
        ]
    )


    running_peak = (
        np.maximum.accumulate(
            equity_curve
        )
    )


    drawdown = (

        equity_curve

        /

        running_peak

        -

        1.0
    )


    growth = float(

        equity_curve[-1]

        -

        1.0
    )


    max_dd = float(
        drawdown.min()
    )


    if max_dd < 0:

        return_to_dd = float(

            growth

            /

            abs(
                max_dd
            )
        )

    else:

        return_to_dd = np.inf


    return {

        "trades":
            int(
                len(r)
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            return_to_dd,
    }


REPLAY_PERFORMANCE = (

    performance_from_returns(

        PAPER_EXECUTION_REPLAY_2026[
            "net_return"
        ]
    )
)


# ============================================================
# 19. EXPECTED PERFORMANCE
# ============================================================

if expected_net_col is not None:

    expected_net_returns = pd.to_numeric(

        EXPECTED_REFERENCE[
            expected_net_col
        ],

        errors="coerce"
    )


    EXPECTED_PERFORMANCE = (

        performance_from_returns(
            expected_net_returns
        )
    )


else:

    # Values from the already-passed historical replay.
    # Used only if historical trade dataframe lacks net_return.

    EXPECTED_PERFORMANCE = {

        "trades":
            300,

        "win_rate":
            np.nan,

        "avg_return":
            0.00011777082708124947,

        "profit_factor":
            1.825282251120899,

        "growth":
            0.03588118888467018,

        "max_dd":
            -0.0051943106534220185,

        "return_to_dd":
            np.nan,
    }


# ============================================================
# 20. PERFORMANCE COMPARISON
# ============================================================

metrics_to_compare = [

    "trades",

    "avg_return",

    "profit_factor",

    "growth",

    "max_dd",
]


performance_rows = []


for metric in metrics_to_compare:

    expected_value = (
        EXPECTED_PERFORMANCE[
            metric
        ]
    )


    replay_value = (
        REPLAY_PERFORMANCE[
            metric
        ]
    )


    try:

        difference = (

            float(
                replay_value
            )

            -

            float(
                expected_value
            )
        )

    except Exception:

        difference = np.nan


    performance_rows.append({

        "metric":
            metric,

        "expected":
            expected_value,

        "replay":
            replay_value,

        "difference":
            difference,
    })


PERFORMANCE_COMPARISON = pd.DataFrame(
    performance_rows
)


print()
print("=" * 110)
print("PERFORMANCE COMPARISON")
print("=" * 110)

display(
    PERFORMANCE_COMPARISON
)


# ============================================================
# 21. FINAL TRADE-LEVEL DIAGNOSTICS
# ============================================================

selected_diagnostic = (

    CONVENTION_DIAGNOSTICS.loc[

        CONVENTION_DIAGNOSTICS[
            "convention"
        ]
        ==
        SELECTED_PRICE_CONVENTION

    ]
    .iloc[0]
)


# ============================================================
# 22. FINAL PERFORMANCE PARITY
# ============================================================

AVG_RETURN_OK = np.isclose(

    REPLAY_PERFORMANCE[
        "avg_return"
    ],

    EXPECTED_PERFORMANCE[
        "avg_return"
    ],

    atol=PERFORMANCE_TOL,

    rtol=0.0
)


PF_OK = np.isclose(

    REPLAY_PERFORMANCE[
        "profit_factor"
    ],

    EXPECTED_PERFORMANCE[
        "profit_factor"
    ],

    atol=PERFORMANCE_TOL,

    rtol=0.0
)


GROWTH_OK = np.isclose(

    REPLAY_PERFORMANCE[
        "growth"
    ],

    EXPECTED_PERFORMANCE[
        "growth"
    ],

    atol=PERFORMANCE_TOL,

    rtol=0.0
)


MAX_DD_OK = np.isclose(

    REPLAY_PERFORMANCE[
        "max_dd"
    ],

    EXPECTED_PERFORMANCE[
        "max_dd"
    ],

    atol=PERFORMANCE_TOL,

    rtol=0.0
)


# ============================================================
# 23. FINAL EXECUTION CHECKS
# ============================================================

EXECUTION_REPLAY_CHECKS = {

    "historical_source_found":

        EXPECTED_SOURCE_NAME
        is not None,


    "expected_trade_count_300":

        len(
            EXPECTED_2026_TRADES
        )
        ==
        300,


    "entry_events_300":

        FINAL_REPLAY_RESULT[
            "entry_count"
        ]
        ==
        300,


    "exit_events_300":

        FINAL_REPLAY_RESULT[
            "exit_count"
        ]
        ==
        300,


    "executed_trades_300":

        len(
            PAPER_EXECUTION_REPLAY_2026
        )
        ==
        300,


    "duplicate_signals_zero":

        DUPLICATE_SIGNALS
        ==
        0,


    "duplicate_entries_zero":

        DUPLICATE_ENTRIES
        ==
        0,


    "overlap_zero":

        OVERLAP_COUNT
        ==
        0,


    "entry_timing_exact":

        ENTRY_TIMING_MISMATCHES
        ==
        0,


    "exit_timing_exact":

        EXIT_TIMING_MISMATCHES
        ==
        0,


    "holding_period_valid":

        INVALID_HOLDING_PERIODS
        ==
        0,


    "execution_state_errors_zero":

        len(
            PAPER_EXECUTION_REPLAY_ERRORS
        )
        ==
        0,


    "flat_at_end":

        bool(
            FINAL_REPLAY_RESULT[
                "flat_at_end"
            ]
        ),


    "price_convention_reproduced":

        bool(
            selected_diagnostic[
                "trade_level_pass"
            ]
        ),


    "avg_return_parity":

        bool(
            AVG_RETURN_OK
        ),


    "profit_factor_parity":

        bool(
            PF_OK
        ),


    "growth_parity":

        bool(
            GROWTH_OK
        ),


    "max_dd_parity":

        bool(
            MAX_DD_OK
        ),
}


EXECUTION_REPLAY_OK = all(

    EXECUTION_REPLAY_CHECKS.values()
)


# ============================================================
# 24. SUMMARY
# ============================================================

print()
print("=" * 110)
print("2026 PAPER EXECUTION REPLAY SUMMARY")
print("=" * 110)

print(
    "Historical source:",
    EXPECTED_SOURCE_NAME
)

print(
    "Execution price convention:",
    SELECTED_PRICE_CONVENTION
)

print(
    "Expected trades:",
    len(
        EXPECTED_2026_TRADES
    )
)

print(
    "Entry events:",
    FINAL_REPLAY_RESULT[
        "entry_count"
    ]
)

print(
    "Exit events:",
    FINAL_REPLAY_RESULT[
        "exit_count"
    ]
)

print(
    "Executed trades:",
    len(
        PAPER_EXECUTION_REPLAY_2026
    )
)

print(
    "Duplicate signals:",
    DUPLICATE_SIGNALS
)

print(
    "Duplicate entries:",
    DUPLICATE_ENTRIES
)

print(
    "Overlap:",
    OVERLAP_COUNT
)

print(
    "Entry timing mismatch:",
    ENTRY_TIMING_MISMATCHES
)

print(
    "Exit timing mismatch:",
    EXIT_TIMING_MISMATCHES
)

print(
    "Execution state errors:",
    len(
        PAPER_EXECUTION_REPLAY_ERRORS
    )
)

print(
    "Flat at end:",
    FINAL_REPLAY_RESULT[
        "flat_at_end"
    ]
)


print()
print("=" * 110)
print("REPLAY PERFORMANCE")
print("=" * 110)


for key, value in (
    REPLAY_PERFORMANCE.items()
):

    if key in [

        "avg_return",
        "growth",
        "max_dd",
    ]:

        print(
            f"{key}: "
            f"{value:.10f} "
            f"({value * 100:.5f}%)"
        )

    else:

        print(
            f"{key}: {value}"
        )


# ============================================================
# 25. FINAL CHECKS
# ============================================================

print()
print("=" * 110)
print("FINAL EXECUTION CHECKS")
print("=" * 110)


for key, value in (
    EXECUTION_REPLAY_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


# ============================================================
# 26. SAVE OUTPUT
# ============================================================

OUTPUT_DIR = (

    Path(
        LIVE_CHAMPION_DIR
    )

    /

    "runtime"

    /

    "paper"

    /

    "historical_execution_replay_2026"
)


OUTPUT_DIR.mkdir(

    parents=True,

    exist_ok=True
)


TRADES_PATH = (

    OUTPUT_DIR

    /

    "paper_execution_replay_2026_trades.csv"
)


PERFORMANCE_PATH = (

    OUTPUT_DIR

    /

    "paper_execution_replay_2026_performance.csv"
)


CONVENTION_PATH = (

    OUTPUT_DIR

    /

    "execution_price_convention_diagnostic.csv"
)


ERRORS_PATH = (

    OUTPUT_DIR

    /

    "paper_execution_replay_2026_errors.csv"
)


DECISION_PATH = (

    OUTPUT_DIR

    /

    "paper_execution_replay_2026_decision.json"
)


PAPER_EXECUTION_REPLAY_2026.to_csv(

    TRADES_PATH,

    index=False
)


PERFORMANCE_COMPARISON.to_csv(

    PERFORMANCE_PATH,

    index=False
)


CONVENTION_DIAGNOSTICS.to_csv(

    CONVENTION_PATH,

    index=False
)


PAPER_EXECUTION_REPLAY_ERRORS.to_csv(

    ERRORS_PATH,

    index=False
)


# ============================================================
# 27. FINAL DECISION
# ============================================================

if EXECUTION_REPLAY_OK:

    EXECUTION_REPLAY_DECISION = (

        "PASS_READY_FOR_STATE_RECOVERY_AND_KILL_SWITCH"
    )

else:

    EXECUTION_REPLAY_DECISION = (

        "FAIL_DO_NOT_START_FORWARD_PAPER_TRADING"
    )


decision_payload = {

    "test":
        "2026_HISTORICAL_PAPER_EXECUTION_REPLAY",

    "champion":
        EXPECTED_CHAMPION,

    "historical_source":
        EXPECTED_SOURCE_NAME,

    "execution_price_convention":
        SELECTED_PRICE_CONVENTION,

    "expected_trades":
        int(
            len(
                EXPECTED_2026_TRADES
            )
        ),

    "executed_trades":
        int(
            len(
                PAPER_EXECUTION_REPLAY_2026
            )
        ),

    "checks": {

        key:
            bool(value)

        for key, value
        in EXECUTION_REPLAY_CHECKS.items()
    },

    "performance": {

        "expected": {

            key:
                (
                    None

                    if (
                        value is None
                        or
                        not np.isfinite(
                            safe_float(
                                value
                            )
                        )
                    )

                    else

                    float(
                        value
                    )
                )

            for key, value
            in EXPECTED_PERFORMANCE.items()
        },

        "replay": {

            key:
                (
                    None

                    if (
                        value is None
                        or
                        not np.isfinite(
                            safe_float(
                                value
                            )
                        )
                    )

                    else

                    float(
                        value
                    )
                )

            for key, value
            in REPLAY_PERFORMANCE.items()
        },
    },

    "decision":
        EXECUTION_REPLAY_DECISION,
}


with open(

    DECISION_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        decision_payload,

        f,

        ensure_ascii=False,

        indent=2
    )


# ============================================================
# 28. SAVE NOTEBOOK VARIABLES
# ============================================================

PRODUCTION_EXECUTION_REPLAY_2026 = (
    PAPER_EXECUTION_REPLAY_2026.copy()
)


PRODUCTION_EXECUTION_REPLAY_PERFORMANCE = (
    REPLAY_PERFORMANCE.copy()
)


PRODUCTION_EXECUTION_REPLAY_EXPECTED_PERFORMANCE = (
    EXPECTED_PERFORMANCE.copy()
)


PRODUCTION_EXECUTION_REPLAY_CHECKS = (
    EXECUTION_REPLAY_CHECKS.copy()
)


PRODUCTION_EXECUTION_REPLAY_OK = bool(
    EXECUTION_REPLAY_OK
)


PRODUCTION_EXECUTION_REPLAY_DECISION = (
    EXECUTION_REPLAY_DECISION
)


PRODUCTION_EXECUTION_PRICE_CONVENTION = (
    SELECTED_PRICE_CONVENTION
)


PRODUCTION_EXECUTION_REPLAY_OUTPUT_DIR = str(
    OUTPUT_DIR
)


# ============================================================
# 29. FINAL REPORT
# ============================================================

print()
print("=" * 110)
print("FINAL DECISION")
print("=" * 110)


print(
    "2026 PAPER EXECUTION REPLAY PASSED:",
    EXECUTION_REPLAY_OK
)


print()

print(
    "FINAL DECISION:",
    EXECUTION_REPLAY_DECISION
)


print()

print(
    "Saved folder:",
    OUTPUT_DIR
)


if EXECUTION_REPLAY_OK:

    print()
    print("=" * 110)
    print("EXECUTION LAYER VALIDATED")
    print("=" * 110)

    print(
        "Historical signals: PASS"
    )

    print(
        "Entry timing: PASS"
    )

    print(
        "Exit timing: PASS"
    )

    print(
        "30-minute holding: PASS"
    )

    print(
        "Overlap protection: PASS"
    )

    print(
        "Execution prices: PASS"
    )

    print(
        "Position sizing: PASS"
    )

    print(
        "Trading costs: PASS"
    )

    print(
        "PnL accounting: PASS"
    )

    print(
        "Equity reproduction: PASS"
    )

    print()

    print(
        "NEXT STEP:"
    )

    print(
        "STATE PERSISTENCE + RESTART RECOVERY "
        "+ KILL SWITCH."
    )


else:

    print()
    print("=" * 110)
    print("STOP")
    print("=" * 110)

    print(
        "Do NOT start Forward Paper Trading."
    )

    print(
        "Inspect only the False checks above."
    )


# ============================================================
# END
# ============================================================


## 元セルindex 77
構文状態：valid


In [ ]:
# ============================================================
# EXECUTION TIMING SEMANTICS DIAGNOSTIC
# ONE-CELL / NON-DESTRUCTIVE / NO TRAINING / NO ORDERS
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 350)
pd.set_option("display.max_rows", 100)

TEST_YEAR = 2026
EXPECTED_TRADES = 300
CHAMPION_NAME = "BASE_PLUS_REGIME"


# ============================================================
# 1. SAFE HELPERS
# ============================================================

def utc_series(values):
    """Always return pandas Series[datetime64[ns, UTC]]."""

    try:
        if isinstance(values, pd.Series):
            s = values.copy().reset_index(drop=True)

        elif isinstance(values, pd.Index):
            s = pd.Series(values.to_numpy())

        elif isinstance(values, np.ndarray):
            s = pd.Series(values)

        elif isinstance(values, (list, tuple)):
            s = pd.Series(list(values))

        else:
            s = pd.Series([values])

        return (
            pd.to_datetime(
                s,
                utc=True,
                errors="coerce"
            )
            .reset_index(drop=True)
        )

    except Exception:
        return pd.Series(
            [],
            dtype="datetime64[ns, UTC]"
        )


def find_exact_col(df, names):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in names:
        key = str(name).strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def overlap_count(entry_times, exit_times):

    e = utc_series(entry_times)
    x = utc_series(exit_times)

    temp = pd.DataFrame({
        "entry": e,
        "exit": x
    }).dropna()

    temp = temp.sort_values("entry").reset_index(drop=True)

    if len(temp) == 0:
        return np.nan

    count = 0
    active_exit = None

    for _, row in temp.iterrows():

        entry = row["entry"]
        exit_ = row["exit"]

        if active_exit is not None and entry < active_exit:
            count += 1

        if active_exit is None or exit_ > active_exit:
            active_exit = exit_

    return int(count)


def holding_stats(entry_times, exit_times):

    e = utc_series(entry_times)
    x = utc_series(exit_times)

    if len(e) != len(x):
        return {
            "valid": 0,
            "median_min": np.nan,
            "min_min": np.nan,
            "max_min": np.nan,
            "exact_30m": 0,
            "overlap": np.nan
        }

    mins = (x - e).dt.total_seconds() / 60.0
    valid = mins.dropna()

    if len(valid) == 0:
        return {
            "valid": 0,
            "median_min": np.nan,
            "min_min": np.nan,
            "max_min": np.nan,
            "exact_30m": 0,
            "overlap": np.nan
        }

    return {
        "valid": int(len(valid)),
        "median_min": float(valid.median()),
        "min_min": float(valid.min()),
        "max_min": float(valid.max()),
        "exact_30m": int(np.isclose(valid, 30.0).sum()),
        "overlap": overlap_count(e, x)
    }


# ============================================================
# 2. FIND THE ACTUAL HISTORICAL TRADE DATAFRAME
# ============================================================

candidate_names = [
    "HISTORICAL_REPLAY_LIVE_TRADES",
    "FINAL_TOURNAMENT_TRADES",
    "CHAMPION_REINTEGRATION_TRADES",
    "CLEAN_BASE_TRADES",
    "PRODUCTION_EXECUTION_REPLAY_2026",
]

source_rows = []
sources = {}

for name in candidate_names:

    obj = globals().get(name)

    if not isinstance(obj, pd.DataFrame):
        source_rows.append({
            "variable": name,
            "exists": False,
            "raw_rows": 0,
            "2026_rows": 0
        })
        continue

    df = obj.copy()

    raw_rows = len(df)

    # Champion filter when present
    feature_col = find_exact_col(
        df,
        ["feature_set", "champion", "strategy"]
    )

    if feature_col is not None:

        vals = (
            df[feature_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        mask = vals.eq(CHAMPION_NAME)

        if mask.any():
            df = df.loc[mask].copy()

    # Explicit year filter when present
    year_col = find_exact_col(
        df,
        ["test_year", "year"]
    )

    if year_col is not None:

        years = pd.to_numeric(
            df[year_col],
            errors="coerce"
        )

        mask = years.eq(TEST_YEAR)

        if mask.any():
            df = df.loc[mask].copy()

    # Signal timestamp
    signal_col = find_exact_col(
        df,
        [
            "signal_time",
            "timestamp",
            "signal_timestamp"
        ]
    )

    if signal_col is not None:
        times = utc_series(df[signal_col])
    else:
        times = utc_series(df.index)

    # Positionally remove bad timestamps
    valid = times.notna().to_numpy()

    if len(valid) == len(df):

        df = df.iloc[valid].copy()
        times = times.iloc[valid].reset_index(drop=True)

        if len(df):

            yr = times.dt.year.eq(TEST_YEAR).to_numpy()

            df = df.iloc[yr].copy()
            times = times.iloc[yr].reset_index(drop=True)

    df = df.reset_index(drop=False)

    if len(times) == len(df):
        df["_diag_signal_time"] = times.to_numpy()

    sources[name] = df

    source_rows.append({
        "variable": name,
        "exists": True,
        "raw_rows": raw_rows,
        "2026_rows": len(df)
    })


SOURCE_DIAGNOSTIC = pd.DataFrame(source_rows)

print("=" * 120)
print("AVAILABLE HISTORICAL SOURCES")
print("=" * 120)

display(SOURCE_DIAGNOSTIC)


# ============================================================
# 3. SELECT SOURCE
# ============================================================

SOURCE_NAME = None
SOURCE = None

# Prefer the historical replay output.
for name in candidate_names:

    df = sources.get(name)

    if not isinstance(df, pd.DataFrame):
        continue

    if len(df) == EXPECTED_TRADES:

        SOURCE_NAME = name
        SOURCE = df.copy()
        break


if SOURCE is None:

    print()
    print("=" * 120)
    print("NO EXACT 300-TRADE SOURCE")
    print("=" * 120)

    print(
        "300-row historical execution source was not found.\n"
        "The table above tells us which object actually exists."
    )

    EXECUTION_TIMING_DIAGNOSTIC_OK = False

else:

    print()
    print("=" * 120)
    print("SELECTED HISTORICAL SOURCE")
    print("=" * 120)

    print("Variable:", SOURCE_NAME)
    print("Rows:", len(SOURCE))


    # ========================================================
    # 4. PRINT ALL COLUMNS
    # ========================================================

    print()
    print("=" * 120)
    print("ALL SOURCE COLUMNS")
    print("=" * 120)

    for i, col in enumerate(SOURCE.columns, start=1):
        print(f"{i:03d}. {col}")


    # ========================================================
    # 5. FIND ALL TIME-LIKE COLUMNS
    # ========================================================

    keywords = (
        "time",
        "date",
        "entry",
        "exit",
        "signal",
        "open",
        "close"
    )

    time_like_cols = []

    for col in SOURCE.columns:

        name_lower = str(col).lower()

        if any(k in name_lower for k in keywords):

            parsed = utc_series(SOURCE[col])

            valid_count = int(parsed.notna().sum())

            if valid_count >= max(10, int(len(SOURCE) * 0.8)):

                time_like_cols.append(col)


    print()
    print("=" * 120)
    print("TIME-LIKE COLUMNS")
    print("=" * 120)

    if time_like_cols:

        for col in time_like_cols:

            parsed = utc_series(SOURCE[col])

            print(
                f"{col}: "
                f"valid={parsed.notna().sum()}/{len(parsed)}, "
                f"min={parsed.min()}, "
                f"max={parsed.max()}"
            )

    else:
        print("No stored time-like columns found.")


    # ========================================================
    # 6. SIGNAL GAP ANALYSIS
    # ========================================================

    if "_diag_signal_time" in SOURCE.columns:

        signal_times = utc_series(
            SOURCE["_diag_signal_time"]
        )

    else:

        sig_col = find_exact_col(
            SOURCE,
            ["signal_time", "timestamp"]
        )

        if sig_col is not None:
            signal_times = utc_series(SOURCE[sig_col])
        else:
            signal_times = utc_series(SOURCE.index)


    signal_times = signal_times.sort_values().reset_index(drop=True)

    gaps = signal_times.diff().dt.total_seconds() / 60.0

    print()
    print("=" * 120)
    print("SIGNAL GAP DIAGNOSTIC")
    print("=" * 120)

    print("Signals:", len(signal_times))
    print("Minimum gap:", gaps.dropna().min(), "minutes")
    print("Median gap:", gaps.dropna().median(), "minutes")
    print("Gaps < 15m:", int((gaps < 15).sum()))
    print("Gaps < 30m:", int((gaps < 30).sum()))
    print("Gaps == 15m:", int(np.isclose(gaps.dropna(), 15).sum()))
    print("Gaps == 30m:", int(np.isclose(gaps.dropna(), 30).sum()))


    # ========================================================
    # 7. DERIVED TIMING CONVENTIONS
    # ========================================================

    derived_rows = []

    conventions = {
        "signal -> signal+30": (
            signal_times,
            signal_times + pd.Timedelta(minutes=30)
        ),

        "signal+15 -> signal+45": (
            signal_times + pd.Timedelta(minutes=15),
            signal_times + pd.Timedelta(minutes=45)
        ),

        "signal close(+15) -> +30 hold": (
            signal_times + pd.Timedelta(minutes=15),
            signal_times + pd.Timedelta(minutes=45)
        )
    }

    for name, (entry, exit_) in conventions.items():

        stats = holding_stats(entry, exit_)

        derived_rows.append({
            "timing_convention": name,
            **stats
        })


    DERIVED_TIMING_DIAGNOSTIC = pd.DataFrame(
        derived_rows
    )

    print()
    print("=" * 120)
    print("DERIVED TIMING CONVENTIONS")
    print("=" * 120)

    display(DERIVED_TIMING_DIAGNOSTIC)


    # ========================================================
    # 8. STORED ENTRY/EXIT COLUMN CANDIDATES
    # ========================================================

    entry_keywords = [
        "entry_time",
        "actual_entry_time",
        "audit_entry_time",
        "open_time",
        "trade_entry_time",
        "entry_timestamp"
    ]

    exit_keywords = [
        "exit_time",
        "actual_exit_time",
        "audit_exit_time",
        "close_time",
        "trade_exit_time",
        "exit_timestamp"
    ]

    entry_candidates = []

    exit_candidates = []


    for col in SOURCE.columns:

        lower = str(col).lower()

        parsed = utc_series(SOURCE[col])

        enough_dates = (
            parsed.notna().sum()
            >=
            int(len(SOURCE) * 0.8)
        )

        if not enough_dates:
            continue

        if (
            "entry" in lower
            or lower in entry_keywords
        ):
            entry_candidates.append(col)

        if (
            "exit" in lower
            or lower in exit_keywords
        ):
            exit_candidates.append(col)


    pair_rows = []

    for entry_col in entry_candidates:

        for exit_col in exit_candidates:

            entry = utc_series(SOURCE[entry_col])
            exit_ = utc_series(SOURCE[exit_col])

            stats = holding_stats(
                entry,
                exit_
            )

            pair_rows.append({
                "entry_col": entry_col,
                "exit_col": exit_col,
                **stats
            })


    STORED_TIMING_DIAGNOSTIC = pd.DataFrame(
        pair_rows
    )


    print()
    print("=" * 120)
    print("STORED ENTRY / EXIT PAIR DIAGNOSTIC")
    print("=" * 120)

    if len(STORED_TIMING_DIAGNOSTIC):

        STORED_TIMING_DIAGNOSTIC = (
            STORED_TIMING_DIAGNOSTIC
            .sort_values(
                [
                    "overlap",
                    "exact_30m"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .reset_index(drop=True)
        )

        display(
            STORED_TIMING_DIAGNOSTIC
        )

    else:

        print(
            "No explicit stored entry/exit timestamp pair "
            "was found in this dataframe."
        )


    # ========================================================
    # 9. FIND A VALID STORED EXECUTION TIMING CONTRACT
    # ========================================================

    RESOLVED_ENTRY_COLUMN = None
    RESOLVED_EXIT_COLUMN = None

    if len(STORED_TIMING_DIAGNOSTIC):

        valid_pairs = STORED_TIMING_DIAGNOSTIC.loc[
            (STORED_TIMING_DIAGNOSTIC["valid"] == EXPECTED_TRADES)
            &
            (STORED_TIMING_DIAGNOSTIC["exact_30m"] == EXPECTED_TRADES)
            &
            (STORED_TIMING_DIAGNOSTIC["overlap"] == 0)
        ].copy()

        if len(valid_pairs):

            RESOLVED_ENTRY_COLUMN = (
                valid_pairs.iloc[0]["entry_col"]
            )

            RESOLVED_EXIT_COLUMN = (
                valid_pairs.iloc[0]["exit_col"]
            )


    # ========================================================
    # 10. FIRST OVERLAP UNDER SIGNAL+15 / +45 ASSUMPTION
    # ========================================================

    derived = pd.DataFrame({
        "row": np.arange(len(signal_times)),
        "signal_time": signal_times,
        "entry_assumed": (
            signal_times
            +
            pd.Timedelta(minutes=15)
        ),
        "exit_assumed": (
            signal_times
            +
            pd.Timedelta(minutes=45)
        )
    })


    derived["previous_exit"] = (
        derived["exit_assumed"].shift(1)
    )

    derived["overlap"] = (
        derived["entry_assumed"]
        <
        derived["previous_exit"]
    )


    first_overlap_positions = (
        derived.index[
            derived["overlap"]
        ].tolist()
    )


    print()
    print("=" * 120)
    print("FIRST OVERLAP UNDER CURRENT ASSUMPTION")
    print("=" * 120)

    if first_overlap_positions:

        first_pos = first_overlap_positions[0]

        start = max(
            0,
            first_pos - 5
        )

        end = min(
            len(derived),
            first_pos + 6
        )

        display(
            derived.iloc[start:end]
        )

        print(
            "First overlap row:",
            first_pos
        )

    else:

        print(
            "No overlap under derived timing."
        )


    # ========================================================
    # 11. SHOW RAW SOURCE ROWS AROUND THE FIRST CONFLICT
    # ========================================================

    print()
    print("=" * 120)
    print("RAW HISTORICAL ROWS AROUND FIRST CONFLICT")
    print("=" * 120)

    if first_overlap_positions:

        first_pos = first_overlap_positions[0]

        start = max(
            0,
            first_pos - 5
        )

        end = min(
            len(SOURCE),
            first_pos + 6
        )

        useful_cols = []

        for col in SOURCE.columns:

            lower = str(col).lower()

            if (
                col == "_diag_signal_time"
                or
                any(
                    k in lower
                    for k in [
                        "signal",
                        "entry",
                        "exit",
                        "side",
                        "position",
                        "size",
                        "return",
                        "price",
                        "confidence",
                        "p_up"
                    ]
                )
            ):
                useful_cols.append(col)

        # avoid huge tables
        useful_cols = useful_cols[:40]

        display(
            SOURCE.iloc[start:end][useful_cols]
        )


    # ========================================================
    # 12. FINAL DIAGNOSTIC DECISION
    # ========================================================

    print()
    print("=" * 120)
    print("FINAL TIMING DIAGNOSTIC")
    print("=" * 120)

    print(
        "Historical source:",
        SOURCE_NAME
    )

    print(
        "Historical rows:",
        len(SOURCE)
    )

    print(
        "Signal gaps <30m:",
        int((gaps < 30).sum())
    )


    if (
        RESOLVED_ENTRY_COLUMN is not None
        and
        RESOLVED_EXIT_COLUMN is not None
    ):

        EXECUTION_TIMING_DIAGNOSTIC_OK = True

        EXECUTION_TIMING_DECISION = (
            "STORED_EXECUTION_TIMING_FOUND"
        )

        print()
        print(
            "RESOLVED ENTRY COLUMN:",
            RESOLVED_ENTRY_COLUMN
        )

        print(
            "RESOLVED EXIT COLUMN:",
            RESOLVED_EXIT_COLUMN
        )

        print()
        print(
            "Decision:",
            EXECUTION_TIMING_DECISION
        )

        print(
            "Stored timestamps give:"
        )

        print(
            " - 300 valid trades"
        )

        print(
            " - exact 30-minute holding"
        )

        print(
            " - zero overlap"
        )

    else:

        EXECUTION_TIMING_DIAGNOSTIC_OK = False

        EXECUTION_TIMING_DECISION = (
            "SOURCE_IS_NOT_DIRECT_EXECUTION_LOG"
        )

        print()
        print(
            "Decision:",
            EXECUTION_TIMING_DECISION
        )

        print()
        print(
            "The 300-row dataframe cannot be treated "
            "directly as 300 sequential paper positions."
        )

        print(
            "We must recover the exact historical "
            "execution-selection rule instead of "
            "silencing OVERLAP_ENTRY."
        )


    # ========================================================
    # 13. SAVE NOTEBOOK VARIABLES
    # ========================================================

    EXECUTION_TIMING_SOURCE_NAME = SOURCE_NAME

    EXECUTION_TIMING_SOURCE = SOURCE.copy()

    EXECUTION_TIMING_SIGNAL_GAPS = gaps.copy()

    EXECUTION_TIMING_DERIVED = (
        DERIVED_TIMING_DIAGNOSTIC.copy()
    )

    EXECUTION_TIMING_STORED = (
        STORED_TIMING_DIAGNOSTIC.copy()
    )

    EXECUTION_TIMING_RESOLVED_ENTRY_COL = (
        RESOLVED_ENTRY_COLUMN
    )

    EXECUTION_TIMING_RESOLVED_EXIT_COL = (
        RESOLVED_EXIT_COLUMN
    )

    EXECUTION_TIMING_DECISION = (
        EXECUTION_TIMING_DECISION
    )


print()
print("=" * 120)
print("DIAGNOSTIC COMPLETE")
print("=" * 120)

print(
    "No strategy/model/Champion parameter was changed."
)

print(
    "No paper/live order was generated."
)


## 元セルindex 78
構文状態：valid


In [ ]:
# ============================================================
# FINAL EXECUTION PRICE SEMANTICS DIAGNOSTIC
# ONE CELL / NON-DESTRUCTIVE / NO TRAINING / NO ORDERS
# ============================================================
#
# Frozen Champion:
#   BASE_PLUS_REGIME
#
# PURPOSE:
#   Recover the exact historical Entry / Exit price semantics
#   from the already validated 300 historical 2026 trades.
#
# IMPORTANT:
#   - No model training
#   - No optimization
#   - No parameter changes
#   - No paper/live orders
#   - No strategy modification
#   - Does NOT crash intentionally; failures are reported cleanly
#
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 350)
pd.set_option("display.max_rows", 100)


# ============================================================
# 0. CONSTANTS
# ============================================================

EXPECTED_CHAMPION = "BASE_PLUS_REGIME"
EXPECTED_TRADES = 300
EXPECTED_YEAR = 2026

BAR_MINUTES = 15

# Historical return reproduction tolerance
STRICT_TOL = 1e-10
PASS_TOL = 1e-8

DEFAULT_COST = 4e-05


# ============================================================
# 1. HELPERS
# ============================================================

def safe_utc_series(values):
    """
    Always returns:
        pandas Series[datetime64[ns, UTC]]

    Prevents the DatetimeIndex .dt error permanently.
    """

    try:

        if isinstance(values, pd.Series):

            s = values.copy().reset_index(drop=True)

        elif isinstance(values, pd.Index):

            s = pd.Series(values.to_numpy())

        elif isinstance(values, np.ndarray):

            s = pd.Series(values)

        elif isinstance(values, (list, tuple)):

            s = pd.Series(list(values))

        else:

            s = pd.Series([values])


        return (
            pd.to_datetime(
                s,
                utc=True,
                errors="coerce"
            )
            .reset_index(drop=True)
        )

    except Exception:

        return pd.Series(
            [],
            dtype="datetime64[ns, UTC]"
        )


def find_col(df, names):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for name in names:

        key = str(name).strip().lower()

        if key in lookup:

            return lookup[key]

    return None


def safe_numeric(values):

    try:

        return (
            pd.to_numeric(
                pd.Series(values).reset_index(drop=True),
                errors="coerce"
            )
        )

    except Exception:

        return pd.Series(
            dtype=float
        )


def finite_max_abs(values):

    x = safe_numeric(values)

    x = x.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(x) == 0:
        return np.nan

    return float(
        x.abs().max()
    )


def finite_mean_abs(values):

    x = safe_numeric(values)

    x = x.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(x) == 0:
        return np.nan

    return float(
        x.abs().mean()
    )


def finite_median_abs(values):

    x = safe_numeric(values)

    x = x.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(x) == 0:
        return np.nan

    return float(
        x.abs().median()
    )


# ============================================================
# 2. MAIN FUNCTION
#
# Everything is wrapped so a diagnostic problem is printed
# instead of creating another long red traceback.
# ============================================================

def run_execution_price_semantics_diagnostic():

    result_object = {

        "ok": False,
        "decision": "NOT_RUN",
        "selected_entry_semantics": None,
        "selected_exit_semantics": None,
        "selected_return_formula": None,
        "selected_cost": None,
    }


    try:

        # ====================================================
        # 3. REQUIRED HISTORICAL TRADE DATA
        # ====================================================

        trades_obj = globals().get(
            "HISTORICAL_REPLAY_LIVE_TRADES"
        )


        if not isinstance(
            trades_obj,
            pd.DataFrame
        ):

            print("=" * 120)
            print("STOP")
            print("=" * 120)

            print(
                "HISTORICAL_REPLAY_LIVE_TRADES "
                "was not found."
            )

            result_object["decision"] = (
                "STOP_MISSING_HISTORICAL_REPLAY_TRADES"
            )

            return result_object


        trades = (
            trades_obj
            .copy()
            .reset_index(drop=True)
        )


        print("=" * 120)
        print("FINAL EXECUTION PRICE SEMANTICS DIAGNOSTIC")
        print("=" * 120)

        print(
            "Historical source:",
            "HISTORICAL_REPLAY_LIVE_TRADES"
        )

        print(
            "Rows:",
            len(trades)
        )


        if len(trades) != EXPECTED_TRADES:

            print()
            print(
                "STOP: Expected 300 historical trades, "
                f"but found {len(trades)}."
            )

            result_object["decision"] = (
                "STOP_HISTORICAL_TRADE_COUNT_MISMATCH"
            )

            return result_object


        # ====================================================
        # 4. REQUIRED TRADE COLUMNS
        # ====================================================

        signal_col = find_col(
            trades,
            ["signal_time"]
        )

        entry_time_col = find_col(
            trades,
            ["entry_time"]
        )

        exit_time_col = find_col(
            trades,
            ["exit_time"]
        )

        side_col = find_col(
            trades,
            ["side"]
        )

        gross_col = find_col(
            trades,
            ["gross_return"]
        )

        net_col = find_col(
            trades,
            ["net_return"]
        )

        size_col = find_col(
            trades,
            ["position_size"]
        )


        required_cols = {

            "signal_time":
                signal_col,

            "entry_time":
                entry_time_col,

            "exit_time":
                exit_time_col,

            "side":
                side_col,

            "gross_return":
                gross_col,

            "net_return":
                net_col,

            "position_size":
                size_col,
        }


        missing = [

            name
            for name, col
            in required_cols.items()

            if col is None
        ]


        if missing:

            print()
            print("=" * 120)
            print("STOP - MISSING REQUIRED COLUMNS")
            print("=" * 120)

            print(
                "Missing:",
                missing
            )

            result_object["decision"] = (
                "STOP_MISSING_REQUIRED_TRADE_COLUMNS"
            )

            return result_object


        # ====================================================
        # 5. NORMALIZE HISTORICAL TRADES
        # ====================================================

        signal_time = safe_utc_series(
            trades[signal_col]
        )

        entry_time = safe_utc_series(
            trades[entry_time_col]
        )

        exit_time = safe_utc_series(
            trades[exit_time_col]
        )


        side = (
            trades[side_col]
            .astype(str)
            .str.strip()
            .str.upper()
            .reset_index(drop=True)
        )


        stored_gross = safe_numeric(
            trades[gross_col]
        )

        stored_net = safe_numeric(
            trades[net_col]
        )

        position_size = safe_numeric(
            trades[size_col]
        )


        normalized = pd.DataFrame({

            "signal_time":
                signal_time,

            "entry_time":
                entry_time,

            "exit_time":
                exit_time,

            "side":
                side,

            "position_size":
                position_size,

            "stored_gross_return":
                stored_gross,

            "stored_net_return":
                stored_net,
        })


        valid_trade_rows = (

            normalized[
                [
                    "signal_time",
                    "entry_time",
                    "exit_time",
                    "side",
                    "position_size",
                    "stored_gross_return",
                    "stored_net_return",
                ]
            ]
            .notna()
            .all(axis=1)
        )


        VALID_ROWS = int(
            valid_trade_rows.sum()
        )


        print()
        print("=" * 120)
        print("HISTORICAL TRADE INTEGRITY")
        print("=" * 120)

        print(
            "Valid complete rows:",
            f"{VALID_ROWS}/{EXPECTED_TRADES}"
        )

        print(
            "BUY trades:",
            int(
                normalized["side"]
                .eq("BUY")
                .sum()
            )
        )

        print(
            "SELL trades:",
            int(
                normalized["side"]
                .eq("SELL")
                .sum()
            )
        )


        if VALID_ROWS != EXPECTED_TRADES:

            print(
                "STOP: Historical trade rows contain "
                "missing values."
            )

            result_object["decision"] = (
                "STOP_INVALID_HISTORICAL_TRADES"
            )

            return result_object


        if not normalized[
            "side"
        ].isin(
            ["BUY", "SELL"]
        ).all():

            print(
                "STOP: Side contains values other than BUY/SELL."
            )

            result_object["decision"] = (
                "STOP_INVALID_SIDE_VALUES"
            )

            return result_object


        # ====================================================
        # 6. RECONFIRM STORED TIMING CONTRACT
        # ====================================================

        hold_minutes = (

            (
                normalized["exit_time"]
                -
                normalized["entry_time"]
            )
            .dt
            .total_seconds()
            /
            60.0
        )


        signal_to_entry = (

            (
                normalized["entry_time"]
                -
                normalized["signal_time"]
            )
            .dt
            .total_seconds()
            /
            60.0
        )


        timing_30_exact = bool(
            np.isclose(
                hold_minutes,
                30.0
            ).all()
        )


        signal_entry_15_exact = bool(
            np.isclose(
                signal_to_entry,
                15.0
            ).all()
        )


        ordered = (
            normalized
            .sort_values("entry_time")
            .reset_index(drop=True)
        )


        previous_exit = (
            ordered["exit_time"]
            .shift(1)
        )


        overlap_count = int(

            (
                ordered["entry_time"]
                <
                previous_exit
            )
            .fillna(False)
            .sum()
        )


        print()
        print("=" * 120)
        print("FROZEN TIMING CONTRACT")
        print("=" * 120)

        print(
            "signal -> entry = 15m:",
            signal_entry_15_exact
        )

        print(
            "entry -> exit = 30m:",
            timing_30_exact
        )

        print(
            "overlap count:",
            overlap_count
        )


        if not (
            signal_entry_15_exact
            and
            timing_30_exact
            and
            overlap_count == 0
        ):

            print(
                "STOP: Stored historical timing contract "
                "does not match the validated policy."
            )

            result_object["decision"] = (
                "STOP_TIMING_CONTRACT_MISMATCH"
            )

            return result_object


        # ====================================================
        # 7. CANONICAL PRICE HISTORY
        # ====================================================

        bars_obj = globals().get(
            "PRODUCTION_CANONICAL_HISTORY"
        )


        if not isinstance(
            bars_obj,
            pd.DataFrame
        ):

            print()
            print("=" * 120)
            print("STOP - CANONICAL HISTORY MISSING")
            print("=" * 120)

            print(
                "PRODUCTION_CANONICAL_HISTORY "
                "was not found."
            )

            result_object["decision"] = (
                "STOP_MISSING_CANONICAL_HISTORY"
            )

            return result_object


        bars = bars_obj.copy()


        # ----------------------------------------------------
        # Determine timestamp source
        # ----------------------------------------------------

        if isinstance(
            bars.index,
            pd.DatetimeIndex
        ):

            bar_time = safe_utc_series(
                bars.index
            )

        else:

            timestamp_col = find_col(

                bars,

                [
                    "timestamp",
                    "datetime",
                    "time"
                ]
            )


            if timestamp_col is None:

                print(
                    "STOP: canonical history timestamp "
                    "could not be identified."
                )

                result_object["decision"] = (
                    "STOP_CANONICAL_TIMESTAMP_NOT_FOUND"
                )

                return result_object


            bar_time = safe_utc_series(
                bars[timestamp_col]
            )


        open_col = find_col(
            bars,
            ["open"]
        )

        high_col = find_col(
            bars,
            ["high"]
        )

        low_col = find_col(
            bars,
            ["low"]
        )

        close_col = find_col(
            bars,
            ["close"]
        )


        if any(
            col is None
            for col in [
                open_col,
                high_col,
                low_col,
                close_col
            ]
        ):

            print(
                "STOP: OHLC columns are missing from "
                "canonical history."
            )

            result_object["decision"] = (
                "STOP_CANONICAL_OHLC_MISSING"
            )

            return result_object


        canonical = pd.DataFrame({

            "timestamp":
                bar_time,

            "open":
                safe_numeric(
                    bars[open_col]
                ),

            "high":
                safe_numeric(
                    bars[high_col]
                ),

            "low":
                safe_numeric(
                    bars[low_col]
                ),

            "close":
                safe_numeric(
                    bars[close_col]
                ),
        })


        canonical = (
            canonical
            .dropna()
            .sort_values("timestamp")
            .drop_duplicates(
                subset=["timestamp"],
                keep="last"
            )
            .set_index("timestamp")
        )


        print()
        print("=" * 120)
        print("CANONICAL HISTORY")
        print("=" * 120)

        print(
            "Rows:",
            f"{len(canonical):,}"
        )

        print(
            "Period:",
            canonical.index.min(),
            "->",
            canonical.index.max()
        )


        # ====================================================
        # 8. PRICE EXTRACTION SEMANTICS
        # ====================================================
        #
        # SAFE:
        #   PREVIOUS_BAR_CLOSE
        #   OPEN_AT_TIME
        #   PREVIOUS_BAR_OPEN
        #
        # DIAGNOSTIC-ONLY:
        #   CLOSE_AT_TIME
        #
        # CLOSE_AT_TIME would use information from the bar
        # AFTER the execution boundary, therefore it cannot
        # become a production contract.
        # ====================================================

        PRICE_MODES = {

            "PREVIOUS_BAR_CLOSE": {
                "offset_minutes": -15,
                "column": "close",
                "production_safe": True,
            },

            "OPEN_AT_TIME": {
                "offset_minutes": 0,
                "column": "open",
                "production_safe": True,
            },

            "PREVIOUS_BAR_OPEN": {
                "offset_minutes": -15,
                "column": "open",
                "production_safe": True,
            },

            "CLOSE_AT_TIME_LOOKAHEAD_DIAGNOSTIC": {
                "offset_minutes": 0,
                "column": "close",
                "production_safe": False,
            },
        }


        def extract_prices(times, mode_name):

            spec = PRICE_MODES[
                mode_name
            ]


            lookup_times = (

                pd.DatetimeIndex(
                    safe_utc_series(
                        times
                    )
                )

                +

                pd.Timedelta(
                    minutes=
                        spec[
                            "offset_minutes"
                        ]
                )
            )


            values = (

                canonical[
                    spec[
                        "column"
                    ]
                ]

                .reindex(
                    lookup_times
                )

                .to_numpy()
            )


            return pd.Series(
                values,
                dtype=float
            )


        # ====================================================
        # 9. RETURN FORMULAS
        # ====================================================
        #
        # Historical expected formula should be:
        #
        # BUY:
        #   exit / entry - 1
        #
        # SELL:
        #   -(exit / entry - 1)
        #
        # We also test inverse SELL only as a diagnostic.
        # ====================================================

        def calculate_gross(
            entry_price,
            exit_price,
            side_values,
            formula_name
        ):

            entry_price = safe_numeric(
                entry_price
            )

            exit_price = safe_numeric(
                exit_price
            )

            side_values = (
                pd.Series(side_values)
                .astype(str)
                .str.upper()
                .reset_index(drop=True)
            )


            normal_return = (

                exit_price
                /
                entry_price
                -
                1.0
            )


            if formula_name == "SIGNED_SIMPLE_RETURN":

                gross = np.where(

                    side_values.eq("BUY"),

                    normal_return,

                    -normal_return
                )


            elif formula_name == "INVERSE_SELL_RETURN_DIAGNOSTIC":

                gross = np.where(

                    side_values.eq("BUY"),

                    normal_return,

                    (
                        entry_price
                        /
                        exit_price
                        -
                        1.0
                    )
                )


            else:

                gross = np.full(
                    len(entry_price),
                    np.nan
                )


            return pd.Series(
                gross,
                dtype=float
            )


        RETURN_FORMULAS = [

            "SIGNED_SIMPLE_RETURN",

            "INVERSE_SELL_RETURN_DIAGNOSTIC",
        ]


        # ====================================================
        # 10. TEST EVERY PRICE COMBINATION
        # ====================================================

        comparison_rows = []

        candidate_cache = {}


        for entry_mode in PRICE_MODES:

            entry_prices = extract_prices(

                normalized[
                    "entry_time"
                ],

                entry_mode
            )


            for exit_mode in PRICE_MODES:

                exit_prices = extract_prices(

                    normalized[
                        "exit_time"
                    ],

                    exit_mode
                )


                for formula_name in RETURN_FORMULAS:

                    calculated_gross = (
                        calculate_gross(

                            entry_prices,

                            exit_prices,

                            normalized[
                                "side"
                            ],

                            formula_name
                        )
                    )


                    gross_diff = (

                        calculated_gross

                        -

                        normalized[
                            "stored_gross_return"
                        ]
                    )


                    valid = (

                        entry_prices.notna()

                        &

                        exit_prices.notna()

                        &

                        calculated_gross.notna()

                        &

                        normalized[
                            "stored_gross_return"
                        ].notna()
                    )


                    valid_count = int(
                        valid.sum()
                    )


                    if valid_count:

                        valid_diff = (
                            gross_diff.loc[
                                valid
                            ]
                        )

                        max_diff = (
                            finite_max_abs(
                                valid_diff
                            )
                        )

                        mean_diff = (
                            finite_mean_abs(
                                valid_diff
                            )
                        )

                        median_diff = (
                            finite_median_abs(
                                valid_diff
                            )
                        )

                        strict_matches = int(

                            (
                                valid_diff.abs()
                                <=
                                STRICT_TOL
                            )
                            .sum()
                        )

                        pass_matches = int(

                            (
                                valid_diff.abs()
                                <=
                                PASS_TOL
                            )
                            .sum()
                        )

                    else:

                        max_diff = np.nan
                        mean_diff = np.nan
                        median_diff = np.nan
                        strict_matches = 0
                        pass_matches = 0


                    production_safe = bool(

                        PRICE_MODES[
                            entry_mode
                        ][
                            "production_safe"
                        ]

                        and

                        PRICE_MODES[
                            exit_mode
                        ][
                            "production_safe"
                        ]

                        and

                        formula_name
                        ==
                        "SIGNED_SIMPLE_RETURN"
                    )


                    key = (
                        entry_mode,
                        exit_mode,
                        formula_name
                    )


                    candidate_cache[
                        key
                    ] = {

                        "entry_prices":
                            entry_prices.copy(),

                        "exit_prices":
                            exit_prices.copy(),

                        "calculated_gross":
                            calculated_gross.copy(),

                        "gross_diff":
                            gross_diff.copy(),
                    }


                    comparison_rows.append({

                        "entry_semantics":
                            entry_mode,

                        "exit_semantics":
                            exit_mode,

                        "return_formula":
                            formula_name,

                        "production_safe":
                            production_safe,

                        "valid_rows":
                            valid_count,

                        "strict_matches":
                            strict_matches,

                        "matches_1e8":
                            pass_matches,

                        "max_abs_diff":
                            max_diff,

                        "mean_abs_diff":
                            mean_diff,

                        "median_abs_diff":
                            median_diff,
                    })


        comparisons = pd.DataFrame(
            comparison_rows
        )


        comparisons = (
            comparisons
            .sort_values(
                [
                    "max_abs_diff",
                    "mean_abs_diff",
                    "production_safe",
                ],
                ascending=[
                    True,
                    True,
                    False,
                ],
                na_position="last"
            )
            .reset_index(drop=True)
        )


        print()
        print("=" * 120)
        print("PRICE SEMANTICS CANDIDATES - BEST RESULTS")
        print("=" * 120)

        display(
            comparisons.head(20)
        )


        # ====================================================
        # 11. SAFE PASSING CANDIDATES
        # ====================================================

        passing = comparisons.loc[

            (
                comparisons[
                    "production_safe"
                ]
                ==
                True
            )

            &

            (
                comparisons[
                    "valid_rows"
                ]
                ==
                EXPECTED_TRADES
            )

            &

            (
                comparisons[
                    "max_abs_diff"
                ]
                <=
                PASS_TOL
            )

        ].copy()


        # ====================================================
        # 12. SELECT SEMANTICS
        # ====================================================
        #
        # Priority if mathematically equivalent:
        #
        # PREVIOUS_BAR_CLOSE
        #
        # because Frozen strategy labels were constructed from
        # fully closed 15-minute bars.
        # ====================================================

        selected_row = None


        if len(passing):

            preferred = passing.loc[

                (
                    passing[
                        "entry_semantics"
                    ]
                    ==
                    "PREVIOUS_BAR_CLOSE"
                )

                &

                (
                    passing[
                        "exit_semantics"
                    ]
                    ==
                    "PREVIOUS_BAR_CLOSE"
                )

                &

                (
                    passing[
                        "return_formula"
                    ]
                    ==
                    "SIGNED_SIMPLE_RETURN"
                )

            ]


            if len(preferred):

                selected_row = (
                    preferred.iloc[0]
                )

            else:

                selected_row = (
                    passing.iloc[0]
                )


        # ====================================================
        # 13. LOOKAHEAD DIAGNOSTIC
        # ====================================================

        all_exact = comparisons.loc[

            (
                comparisons[
                    "valid_rows"
                ]
                ==
                EXPECTED_TRADES
            )

            &

            (
                comparisons[
                    "max_abs_diff"
                ]
                <=
                PASS_TOL
            )

        ].copy()


        only_lookahead_passes = bool(

            len(all_exact) > 0

            and

            len(passing) == 0
        )


        # ====================================================
        # 14. NO VALID PRODUCTION SEMANTICS
        # ====================================================

        if selected_row is None:

            print()
            print("=" * 120)
            print("FINAL PRICE SEMANTICS DECISION")
            print("=" * 120)


            if only_lookahead_passes:

                print(
                    "FAIL: Only a lookahead price convention "
                    "can reproduce historical returns."
                )

                decision = (
                    "FAIL_LOOKAHEAD_PRICE_SEMANTICS_DETECTED"
                )

            else:

                print(
                    "FAIL: No production-safe price convention "
                    "reproduces all 300 historical gross returns."
                )

                decision = (
                    "FAIL_PRICE_SEMANTICS_NOT_REPRODUCED"
                )


            print()
            print(
                "Do NOT modify the Champion."
            )

            print(
                "Do NOT start Forward Paper Trading."
            )


            result_object.update({

                "ok": False,

                "decision":
                    decision,

                "comparisons":
                    comparisons,
            })


            globals()[
                "EXECUTION_PRICE_SEMANTICS_RESULTS"
            ] = comparisons.copy()


            globals()[
                "EXECUTION_PRICE_SEMANTICS_OK"
            ] = False


            globals()[
                "EXECUTION_PRICE_SEMANTICS_DECISION"
            ] = decision


            return result_object


        # ====================================================
        # 15. SELECTED SEMANTICS
        # ====================================================

        selected_entry = str(
            selected_row[
                "entry_semantics"
            ]
        )

        selected_exit = str(
            selected_row[
                "exit_semantics"
            ]
        )

        selected_formula = str(
            selected_row[
                "return_formula"
            ]
        )


        selected_key = (

            selected_entry,

            selected_exit,

            selected_formula,
        )


        selected_data = (
            candidate_cache[
                selected_key
            ]
        )


        # ====================================================
        # 16. COST / SIZING ACCOUNTING DIAGNOSTIC
        # ====================================================
        #
        # Expected:
        #
        # net_return =
        #     gross_return * position_size
        #     - cost
        #
        # ====================================================

        calculated_gross = (
            selected_data[
                "calculated_gross"
            ]
        )


        implied_cost = (

            calculated_gross

            *

            normalized[
                "position_size"
            ]

            -

            normalized[
                "stored_net_return"
            ]
        )


        implied_cost_clean = (

            implied_cost

            .replace(
                [np.inf, -np.inf],
                np.nan
            )

            .dropna()
        )


        if len(
            implied_cost_clean
        ):

            median_implied_cost = float(
                implied_cost_clean.median()
            )

            max_cost_deviation = float(

                (
                    implied_cost_clean
                    -
                    median_implied_cost
                )
                .abs()
                .max()
            )

        else:

            median_implied_cost = np.nan
            max_cost_deviation = np.nan


        # ----------------------------------------------------
        # Frozen contract cost if available
        # ----------------------------------------------------

        contract_cost = np.nan


        paper_contract = globals().get(
            "PRODUCTION_PAPER_ENGINE_CONTRACT"
        )


        if isinstance(
            paper_contract,
            dict
        ):

            try:

                contract_cost = float(

                    paper_contract.get(
                        "base_cost_return",
                        DEFAULT_COST
                    )
                )

            except Exception:

                contract_cost = DEFAULT_COST


        if not np.isfinite(
            contract_cost
        ):

            contract_cost = DEFAULT_COST


        reconstructed_net = (

            calculated_gross

            *

            normalized[
                "position_size"
            ]

            -

            contract_cost
        )


        net_diff = (

            reconstructed_net

            -

            normalized[
                "stored_net_return"
            ]
        )


        max_net_diff = finite_max_abs(
            net_diff
        )


        COST_MATCH = bool(

            np.isfinite(
                median_implied_cost
            )

            and

            abs(
                median_implied_cost
                -
                contract_cost
            )
            <=
            PASS_TOL
        )


        COST_STABLE = bool(

            np.isfinite(
                max_cost_deviation
            )

            and

            max_cost_deviation
            <=
            PASS_TOL
        )


        NET_PARITY = bool(

            np.isfinite(
                max_net_diff
            )

            and

            max_net_diff
            <=
            PASS_TOL
        )


        # ====================================================
        # 17. PRICE EQUIVALENCE DIAGNOSTIC
        # ====================================================

        previous_entry = extract_prices(

            normalized[
                "entry_time"
            ],

            "PREVIOUS_BAR_CLOSE"
        )


        open_entry = extract_prices(

            normalized[
                "entry_time"
            ],

            "OPEN_AT_TIME"
        )


        previous_exit = extract_prices(

            normalized[
                "exit_time"
            ],

            "PREVIOUS_BAR_CLOSE"
        )


        open_exit = extract_prices(

            normalized[
                "exit_time"
            ],

            "OPEN_AT_TIME"
        )


        entry_open_gap_count = int(

            (
                (
                    previous_entry
                    -
                    open_entry
                )
                .abs()
                >
                1e-12
            )
            .fillna(False)
            .sum()
        )


        exit_open_gap_count = int(

            (
                (
                    previous_exit
                    -
                    open_exit
                )
                .abs()
                >
                1e-12
            )
            .fillna(False)
            .sum()
        )


        # ====================================================
        # 18. TRADE SAMPLE
        # ====================================================

        selected_trade_table = normalized.copy()


        selected_trade_table[
            "entry_price"
        ] = selected_data[
            "entry_prices"
        ]


        selected_trade_table[
            "exit_price"
        ] = selected_data[
            "exit_prices"
        ]


        selected_trade_table[
            "reconstructed_gross"
        ] = calculated_gross


        selected_trade_table[
            "gross_diff"
        ] = selected_data[
            "gross_diff"
        ]


        selected_trade_table[
            "reconstructed_net"
        ] = reconstructed_net


        selected_trade_table[
            "net_diff"
        ] = net_diff


        selected_trade_table[
            "implied_cost"
        ] = implied_cost


        # ====================================================
        # 19. FINAL CHECKS
        # ====================================================

        GROSS_PARITY = bool(

            float(
                selected_row[
                    "max_abs_diff"
                ]
            )
            <=
            PASS_TOL
        )


        TIMING_PARITY = bool(

            signal_entry_15_exact

            and

            timing_30_exact

            and

            overlap_count == 0
        )


        PRODUCTION_SAFE = bool(

            selected_row[
                "production_safe"
            ]
        )


        FINAL_OK = bool(

            GROSS_PARITY

            and

            NET_PARITY

            and

            COST_MATCH

            and

            COST_STABLE

            and

            TIMING_PARITY

            and

            PRODUCTION_SAFE
        )


        # ====================================================
        # 20. REPORT
        # ====================================================

        print()
        print("=" * 120)
        print("SELECTED EXECUTION PRICE SEMANTICS")
        print("=" * 120)

        print(
            "Entry:",
            selected_entry
        )

        print(
            "Exit:",
            selected_exit
        )

        print(
            "Return formula:",
            selected_formula
        )

        print(
            "Production safe:",
            PRODUCTION_SAFE
        )

        print()
        print(
            "Historical gross max diff:",
            float(
                selected_row[
                    "max_abs_diff"
                ]
            )
        )

        print(
            "Historical gross mean diff:",
            float(
                selected_row[
                    "mean_abs_diff"
                ]
            )
        )

        print(
            "Gross matches <=1e-8:",
            int(
                selected_row[
                    "matches_1e8"
                ]
            ),
            "/",
            EXPECTED_TRADES
        )


        print()
        print("=" * 120)
        print("COST / SIZING ACCOUNTING")
        print("=" * 120)

        print(
            "Frozen contract cost:",
            contract_cost
        )

        print(
            "Median implied cost:",
            median_implied_cost
        )

        print(
            "Max implied-cost deviation:",
            max_cost_deviation
        )

        print(
            "Max net-return difference:",
            max_net_diff
        )

        print()
        print(
            "Cost matches:",
            COST_MATCH
        )

        print(
            "Cost stable:",
            COST_STABLE
        )

        print(
            "Net return parity:",
            NET_PARITY
        )


        print()
        print("=" * 120)
        print("BOUNDARY PRICE EQUIVALENCE")
        print("=" * 120)

        print(
            "Entry boundaries where "
            "previous close != current open:",
            entry_open_gap_count
        )

        print(
            "Exit boundaries where "
            "previous close != current open:",
            exit_open_gap_count
        )


        print()
        print("=" * 120)
        print("WORST GROSS-RETURN DIFFERENCES")
        print("=" * 120)

        display(

            selected_trade_table

            .assign(
                abs_gross_diff=
                    selected_trade_table[
                        "gross_diff"
                    ].abs()
            )

            .sort_values(
                "abs_gross_diff",
                ascending=False
            )

            .head(15)
        )


        # ====================================================
        # 21. SAVE NOTEBOOK VARIABLES
        # ====================================================

        globals()[
            "EXECUTION_PRICE_SEMANTICS_RESULTS"
        ] = comparisons.copy()


        globals()[
            "EXECUTION_PRICE_SELECTED_TRADES"
        ] = selected_trade_table.copy()


        globals()[
            "EXECUTION_PRICE_ENTRY_MODE"
        ] = selected_entry


        globals()[
            "EXECUTION_PRICE_EXIT_MODE"
        ] = selected_exit


        globals()[
            "EXECUTION_PRICE_RETURN_FORMULA"
        ] = selected_formula


        globals()[
            "EXECUTION_PRICE_BASE_COST"
        ] = float(
            contract_cost
        )


        globals()[
            "EXECUTION_PRICE_SEMANTICS_OK"
        ] = FINAL_OK


        # ====================================================
        # 22. SAVE FILES IF CHAMPION DIRECTORY EXISTS
        # ====================================================

        saved_dir = None


        live_dir = globals().get(
            "LIVE_CHAMPION_DIR"
        )


        if live_dir is not None:

            try:

                saved_dir = (

                    Path(
                        live_dir
                    )

                    /

                    "runtime"

                    /

                    "paper"

                    /

                    "execution_price_semantics"
                )


                saved_dir.mkdir(
                    parents=True,
                    exist_ok=True
                )


                comparisons.to_csv(

                    saved_dir
                    /
                    "price_semantics_candidates.csv",

                    index=False
                )


                selected_trade_table.to_csv(

                    saved_dir
                    /
                    "selected_price_semantics_trades.csv",

                    index=False
                )


                manifest = {

                    "champion":
                        EXPECTED_CHAMPION,

                    "year":
                        EXPECTED_YEAR,

                    "historical_trades":
                        EXPECTED_TRADES,

                    "entry_semantics":
                        selected_entry,

                    "exit_semantics":
                        selected_exit,

                    "return_formula":
                        selected_formula,

                    "base_cost":
                        float(
                            contract_cost
                        ),

                    "gross_max_abs_diff":
                        float(
                            selected_row[
                                "max_abs_diff"
                            ]
                        ),

                    "net_max_abs_diff":
                        float(
                            max_net_diff
                        ),

                    "timing_parity":
                        bool(
                            TIMING_PARITY
                        ),

                    "gross_parity":
                        bool(
                            GROSS_PARITY
                        ),

                    "net_parity":
                        bool(
                            NET_PARITY
                        ),

                    "cost_match":
                        bool(
                            COST_MATCH
                        ),

                    "cost_stable":
                        bool(
                            COST_STABLE
                        ),

                    "production_safe":
                        bool(
                            PRODUCTION_SAFE
                        ),

                    "passed":
                        bool(
                            FINAL_OK
                        ),
                }


                with open(

                    saved_dir
                    /
                    "execution_price_semantics.json",

                    "w",

                    encoding="utf-8"

                ) as f:

                    json.dump(

                        manifest,

                        f,

                        ensure_ascii=False,

                        indent=2
                    )


            except Exception as save_error:

                print()
                print(
                    "WARNING: Diagnostic succeeded but "
                    "file saving failed:"
                )

                print(
                    type(save_error).__name__,
                    str(save_error)
                )


        # ====================================================
        # 23. FINAL DECISION
        # ====================================================

        print()
        print("=" * 120)
        print("FINAL EXECUTION PRICE SEMANTICS DECISION")
        print("=" * 120)


        if FINAL_OK:

            decision = (
                "PASS_EXECUTION_PRICE_SEMANTICS_FROZEN"
            )

            print(
                "EXECUTION PRICE SEMANTICS PASSED: True"
            )

            print()
            print(
                "FINAL DECISION:",
                decision
            )

            print()
            print(
                "Historical Entry/Exit prices can now be "
                "reproduced without changing the Champion."
            )

            print(
                "300 historical gross returns reproduce."
            )

            print(
                "Position sizing / cost / net returns reproduce."
            )

            print()
            print(
                "NEXT STEP:"
            )

            print(
                "BUILD FINAL 2026 PAPER EXECUTION REPLAY "
                "USING THESE FROZEN PRICE SEMANTICS."
            )

        else:

            decision = (
                "FAIL_DO_NOT_START_FORWARD_PAPER_TRADING"
            )

            print(
                "EXECUTION PRICE SEMANTICS PASSED: False"
            )

            print()
            print(
                "FINAL DECISION:",
                decision
            )

            print()
            print(
                "Do NOT change the Champion."
            )

            print(
                "Do NOT start Forward Paper Trading."
            )

            print(
                "Use the diagnostics above to identify "
                "the remaining accounting mismatch."
            )


        globals()[
            "EXECUTION_PRICE_SEMANTICS_DECISION"
        ] = decision


        result_object.update({

            "ok":
                FINAL_OK,

            "decision":
                decision,

            "selected_entry_semantics":
                selected_entry,

            "selected_exit_semantics":
                selected_exit,

            "selected_return_formula":
                selected_formula,

            "selected_cost":
                float(
                    contract_cost
                ),

            "gross_max_abs_diff":
                float(
                    selected_row[
                        "max_abs_diff"
                    ]
                ),

            "net_max_abs_diff":
                float(
                    max_net_diff
                ),

            "comparisons":
                comparisons,

            "trade_table":
                selected_trade_table,

            "saved_directory":
                (
                    str(saved_dir)
                    if saved_dir is not None
                    else None
                ),
        })


        return result_object


    except Exception as exc:

        # ====================================================
        # SAFETY CATCH
        #
        # The diagnostic reports unexpected implementation
        # issues without producing another giant traceback.
        # ====================================================

        print()
        print("=" * 120)
        print("DIAGNOSTIC STOPPED SAFELY")
        print("=" * 120)

        print(
            "Exception type:",
            type(exc).__name__
        )

        print(
            "Message:",
            str(exc)
        )

        print()
        print(
            "No model/strategy/Champion parameter "
            "was modified."
        )

        print(
            "No paper/live order was generated."
        )


        decision = (
            "STOP_DIAGNOSTIC_IMPLEMENTATION_ERROR"
        )


        globals()[
            "EXECUTION_PRICE_SEMANTICS_OK"
        ] = False


        globals()[
            "EXECUTION_PRICE_SEMANTICS_DECISION"
        ] = decision


        result_object.update({

            "ok":
                False,

            "decision":
                decision,

            "exception_type":
                type(exc).__name__,

            "exception_message":
                str(exc),
        })


        return result_object


# ============================================================
# 24. RUN ONCE
# ============================================================

EXECUTION_PRICE_DIAGNOSTIC_REPORT = (
    run_execution_price_semantics_diagnostic()
)


print()
print("=" * 120)
print("CELL COMPLETE")
print("=" * 120)

print(
    "Decision:",
    EXECUTION_PRICE_DIAGNOSTIC_REPORT.get(
        "decision"
    )
)

print(
    "Passed:",
    EXECUTION_PRICE_DIAGNOSTIC_REPORT.get(
        "ok"
    )
)

# ============================================================
# END
# ============================================================


## 元セルindex 79
構文状態：valid


In [ ]:
# ============================================================
# ONE-SHOT EXECUTION PRICE SEMANTICS RECOVERY
# schema fix + timing check + price semantics
#
# NO TRAINING
# NO OPTIMIZATION
# NO ORDERS
# NO CHAMPION MODIFICATION
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 50)

EXPECTED_TRADES = 300
PASS_TOL = 1e-9


# ============================================================
# HELPERS
# ============================================================

def to_utc_series(x):
    try:
        if isinstance(x, pd.Series):
            s = x.reset_index(drop=True)
        elif isinstance(x, pd.Index):
            s = pd.Series(x.to_numpy())
        else:
            s = pd.Series(x)

        return pd.to_datetime(
            s,
            utc=True,
            errors="coerce"
        ).reset_index(drop=True)

    except Exception:
        return pd.Series([], dtype="datetime64[ns, UTC]")


def num_series(x):
    return pd.to_numeric(
        pd.Series(x).reset_index(drop=True),
        errors="coerce"
    )


def find_col(df, candidates):
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]

    return None


def stop(msg, decision):
    print()
    print("=" * 110)
    print("STOP")
    print("=" * 110)
    print(msg)
    print("Decision:", decision)

    globals()["EXECUTION_PRICE_SEMANTICS_OK"] = False
    globals()["EXECUTION_PRICE_SEMANTICS_DECISION"] = decision

    return {
        "ok": False,
        "decision": decision
    }


# ============================================================
# MAIN
# ============================================================

def run_one_shot_price_semantics():

    try:

        print("=" * 110)
        print("ONE-SHOT EXECUTION PRICE SEMANTICS RECOVERY")
        print("=" * 110)

        # ----------------------------------------------------
        # 1. Historical 300 trades
        # ----------------------------------------------------

        obj = globals().get("HISTORICAL_REPLAY_LIVE_TRADES")

        if not isinstance(obj, pd.DataFrame):
            return stop(
                "HISTORICAL_REPLAY_LIVE_TRADES がありません。",
                "STOP_MISSING_HISTORICAL_TRADES"
            )

        trades = obj.copy()

        print("Historical rows:", len(trades))

        if len(trades) != EXPECTED_TRADES:
            return stop(
                f"300取引を想定していますが {len(trades)} 行です。",
                "STOP_WRONG_TRADE_COUNT"
            )


        # ----------------------------------------------------
        # 2. Recover signal_time robustly
        # ----------------------------------------------------

        if "signal_time" in trades.columns:

            signal = to_utc_series(
                trades["signal_time"]
            )
            signal_source = "signal_time column"

        elif "_diag_signal_time" in trades.columns:

            signal = to_utc_series(
                trades["_diag_signal_time"]
            )
            signal_source = "_diag_signal_time column"

        elif isinstance(trades.index, pd.DatetimeIndex):

            signal = to_utc_series(
                trades.index
            )
            signal_source = "DatetimeIndex"

        else:

            # Try index as datetime only if every row parses.
            candidate = to_utc_series(
                trades.index
            )

            if (
                len(candidate) == len(trades)
                and candidate.notna().all()
            ):
                signal = candidate
                signal_source = "parsed dataframe index"
            else:
                return stop(
                    "signal_time を列・_diag_signal_time・indexから復元できません。",
                    "STOP_SIGNAL_TIME_NOT_FOUND"
                )


        # Reset AFTER recovering the index timestamps.
        trades = trades.reset_index(drop=True)

        trades["signal_time"] = signal.to_numpy()


        # ----------------------------------------------------
        # 3. Required columns
        # ----------------------------------------------------

        entry_col = find_col(
            trades,
            ["entry_time"]
        )

        exit_col = find_col(
            trades,
            ["exit_time"]
        )

        side_col = find_col(
            trades,
            ["side"]
        )

        gross_col = find_col(
            trades,
            ["gross_return"]
        )


        missing = []

        if entry_col is None:
            missing.append("entry_time")

        if exit_col is None:
            missing.append("exit_time")

        if side_col is None:
            missing.append("side")

        if gross_col is None:
            missing.append("gross_return")


        if missing:
            return stop(
                f"必要列がありません: {missing}",
                "STOP_REQUIRED_COLUMNS_MISSING"
            )


        trades["entry_time"] = to_utc_series(
            trades[entry_col]
        ).to_numpy()

        trades["exit_time"] = to_utc_series(
            trades[exit_col]
        ).to_numpy()

        trades["side"] = (
            trades[side_col]
            .astype(str)
            .str.upper()
            .str.strip()
        )

        trades["stored_gross_return"] = num_series(
            trades[gross_col]
        ).to_numpy()


        # ----------------------------------------------------
        # 4. Integrity
        # ----------------------------------------------------

        if trades[
            [
                "signal_time",
                "entry_time",
                "exit_time",
                "stored_gross_return"
            ]
        ].isna().any().any():

            return stop(
                "Historical trades に欠損値があります。",
                "STOP_TRADE_NAN"
            )


        if not trades["side"].isin(["BUY", "SELL"]).all():

            return stop(
                "side に BUY/SELL 以外があります。",
                "STOP_INVALID_SIDE"
            )


        # ----------------------------------------------------
        # 5. Timing contract
        # ----------------------------------------------------

        signal_to_entry = (
            trades["entry_time"]
            -
            trades["signal_time"]
        ).dt.total_seconds() / 60.0

        holding = (
            trades["exit_time"]
            -
            trades["entry_time"]
        ).dt.total_seconds() / 60.0


        ordered = (
            trades
            .sort_values("entry_time")
            .reset_index(drop=True)
        )

        overlap = (
            ordered["entry_time"]
            <
            ordered["exit_time"].shift(1)
        ).fillna(False)


        signal15 = bool(
            np.isclose(
                signal_to_entry,
                15.0
            ).all()
        )

        hold30 = bool(
            np.isclose(
                holding,
                30.0
            ).all()
        )

        overlap_count = int(
            overlap.sum()
        )


        print()
        print("=" * 110)
        print("TIMING CONTRACT")
        print("=" * 110)

        print("signal source:", signal_source)
        print("signal -> entry = 15m:", signal15)
        print("entry -> exit = 30m:", hold30)
        print("overlap:", overlap_count)


        if not (
            signal15
            and hold30
            and overlap_count == 0
        ):
            return stop(
                "確定済みHistorical Timing Contractと一致しません。",
                "STOP_TIMING_CONTRACT_MISMATCH"
            )


        # Save corrected schema
        globals()["HISTORICAL_REPLAY_LIVE_TRADES"] = (
            trades.copy()
        )


        # ----------------------------------------------------
        # 6. Locate EXACT canonical history
        # ----------------------------------------------------

        canonical_candidates = [
            "PRODUCTION_CANONICAL_HISTORY",
            "CANONICAL_HISTORY",
            "bars_rebuilt_clean",
        ]


        selected_name = None
        source = None


        for name in canonical_candidates:

            x = globals().get(name)

            if not isinstance(x, pd.DataFrame):
                continue

            # Prefer the exact known clean-series size.
            if len(x) == 265905:
                selected_name = name
                source = x.copy()
                break


        if source is None:

            print()
            print("Canonical candidates found:")

            for name in canonical_candidates:
                x = globals().get(name)

                if isinstance(x, pd.DataFrame):
                    print(
                        f"  {name}: {len(x):,} rows"
                    )

            return stop(
                "265,905行のProduction Canonical Historyを特定できません。"
                " 非Canonical barsへ自動fallbackはしません。",
                "STOP_EXACT_CANONICAL_HISTORY_NOT_FOUND"
            )


        print()
        print("=" * 110)
        print("CANONICAL HISTORY")
        print("=" * 110)

        print("Source:", selected_name)
        print("Rows:", f"{len(source):,}")


        # ----------------------------------------------------
        # 7. Normalize canonical OHLC
        # ----------------------------------------------------

        if isinstance(source.index, pd.DatetimeIndex):

            ts = to_utc_series(
                source.index
            )

        else:

            tcol = find_col(
                source,
                [
                    "timestamp",
                    "datetime",
                    "time"
                ]
            )

            if tcol is None:
                return stop(
                    "Canonical Historyのtimestamp列を特定できません。",
                    "STOP_CANONICAL_TIMESTAMP_MISSING"
                )

            ts = to_utc_series(
                source[tcol]
            )


        ocol = find_col(source, ["open"])
        hcol = find_col(source, ["high"])
        lcol = find_col(source, ["low"])
        ccol = find_col(source, ["close"])


        if any(
            c is None
            for c in [
                ocol,
                hcol,
                lcol,
                ccol
            ]
        ):
            return stop(
                "Canonical HistoryにOHLC列がありません。",
                "STOP_CANONICAL_OHLC_MISSING"
            )


        canonical = pd.DataFrame({
            "timestamp": ts,
            "open": num_series(source[ocol]),
            "high": num_series(source[hcol]),
            "low": num_series(source[lcol]),
            "close": num_series(source[ccol]),
        })


        canonical = (
            canonical
            .dropna()
            .sort_values("timestamp")
            .drop_duplicates(
                "timestamp",
                keep="last"
            )
            .set_index("timestamp")
        )


        print(
            "Period:",
            canonical.index.min(),
            "->",
            canonical.index.max()
        )


        # ----------------------------------------------------
        # 8. Price lookup definitions
        # ----------------------------------------------------

        price_modes = {

            # Price known exactly at boundary t
            "PREVIOUS_BAR_CLOSE": (
                -15,
                "close"
            ),

            # Opening price of bar at boundary t
            "OPEN_AT_TIME": (
                0,
                "open"
            ),

            # Diagnostic
            "PREVIOUS_BAR_OPEN": (
                -15,
                "open"
            ),
        }


        def get_price(times, mode):

            offset_min, column = (
                price_modes[mode]
            )

            lookup_times = (
                pd.DatetimeIndex(
                    to_utc_series(times)
                )
                +
                pd.Timedelta(
                    minutes=offset_min
                )
            )

            arr = (
                canonical[column]
                .reindex(lookup_times)
                .to_numpy()
            )

            return pd.Series(
                arr,
                dtype=float
            )


        # ----------------------------------------------------
        # 9. Calculate direction-adjusted gross return
        # ----------------------------------------------------

        def calc_return(entry, exit_, side):

            raw = (
                exit_
                /
                entry
                -
                1.0
            )

            return pd.Series(
                np.where(
                    side.eq("BUY"),
                    raw,
                    -raw
                ),
                dtype=float
            )


        # ----------------------------------------------------
        # 10. Test combinations
        # ----------------------------------------------------

        rows = []
        cache = {}


        for entry_mode in price_modes:

            entry_price = get_price(
                trades["entry_time"],
                entry_mode
            )


            for exit_mode in price_modes:

                exit_price = get_price(
                    trades["exit_time"],
                    exit_mode
                )


                recreated = calc_return(
                    entry_price,
                    exit_price,
                    trades["side"].reset_index(drop=True)
                )


                expected = (
                    trades[
                        "stored_gross_return"
                    ]
                    .reset_index(drop=True)
                )


                diff = (
                    recreated
                    -
                    expected
                )


                valid = (
                    entry_price.notna()
                    &
                    exit_price.notna()
                    &
                    recreated.notna()
                    &
                    expected.notna()
                )


                valid_n = int(
                    valid.sum()
                )


                if valid_n:

                    d = diff.loc[valid]

                    max_diff = float(
                        d.abs().max()
                    )

                    mean_diff = float(
                        d.abs().mean()
                    )

                    matches = int(
                        (
                            d.abs()
                            <=
                            PASS_TOL
                        ).sum()
                    )

                else:

                    max_diff = np.nan
                    mean_diff = np.nan
                    matches = 0


                rows.append({

                    "entry_semantics":
                        entry_mode,

                    "exit_semantics":
                        exit_mode,

                    "valid_rows":
                        valid_n,

                    "matches_1e9":
                        matches,

                    "max_abs_diff":
                        max_diff,

                    "mean_abs_diff":
                        mean_diff,
                })


                cache[
                    (
                        entry_mode,
                        exit_mode
                    )
                ] = {
                    "entry": entry_price,
                    "exit": exit_price,
                    "recreated": recreated,
                    "diff": diff
                }


        results = (
            pd.DataFrame(rows)
            .sort_values(
                [
                    "max_abs_diff",
                    "mean_abs_diff"
                ],
                na_position="last"
            )
            .reset_index(drop=True)
        )


        print()
        print("=" * 110)
        print("EXECUTION PRICE CANDIDATES")
        print("=" * 110)

        display(results)


        # ----------------------------------------------------
        # 11. Exact candidates
        # ----------------------------------------------------

        passing = results.loc[
            (results["valid_rows"] == EXPECTED_TRADES)
            &
            (results["max_abs_diff"] <= PASS_TOL)
        ].copy()


        if len(passing) == 0:

            globals()[
                "EXECUTION_PRICE_SEMANTICS_RESULTS"
            ] = results.copy()

            return stop(
                "300件すべてのgross_returnを再現する価格仕様が見つかりません。",
                "FAIL_EXECUTION_PRICE_SEMANTICS_NOT_REPRODUCED"
            )


        # Prefer the historical closed-bar convention.
        preferred = passing.loc[
            (passing["entry_semantics"] == "PREVIOUS_BAR_CLOSE")
            &
            (passing["exit_semantics"] == "PREVIOUS_BAR_CLOSE")
        ]


        if len(preferred):
            best = preferred.iloc[0]
        else:
            best = passing.iloc[0]


        entry_mode = best[
            "entry_semantics"
        ]

        exit_mode = best[
            "exit_semantics"
        ]


        detail = cache[
            (
                entry_mode,
                exit_mode
            )
        ]


        # ----------------------------------------------------
        # 12. Reproduction table
        # ----------------------------------------------------

        replay = trades[
            [
                "signal_time",
                "entry_time",
                "exit_time",
                "side",
                "stored_gross_return"
            ]
        ].copy()


        replay["entry_price"] = (
            detail["entry"].to_numpy()
        )

        replay["exit_price"] = (
            detail["exit"].to_numpy()
        )

        replay["recreated_gross_return"] = (
            detail["recreated"].to_numpy()
        )

        replay["difference"] = (
            detail["diff"].to_numpy()
        )


        # ----------------------------------------------------
        # 13. Boundary-equivalence check
        # ----------------------------------------------------

        entry_prev_close = get_price(
            trades["entry_time"],
            "PREVIOUS_BAR_CLOSE"
        )

        entry_open = get_price(
            trades["entry_time"],
            "OPEN_AT_TIME"
        )

        exit_prev_close = get_price(
            trades["exit_time"],
            "PREVIOUS_BAR_CLOSE"
        )

        exit_open = get_price(
            trades["exit_time"],
            "OPEN_AT_TIME"
        )


        entry_boundary_diff = int(
            (
                (
                    entry_prev_close
                    -
                    entry_open
                ).abs()
                >
                1e-12
            ).sum()
        )


        exit_boundary_diff = int(
            (
                (
                    exit_prev_close
                    -
                    exit_open
                ).abs()
                >
                1e-12
            ).sum()
        )


        # ----------------------------------------------------
        # 14. Save
        # ----------------------------------------------------

        globals()[
            "EXECUTION_PRICE_SEMANTICS_RESULTS"
        ] = results.copy()

        globals()[
            "EXECUTION_PRICE_REPLAY"
        ] = replay.copy()

        globals()[
            "EXECUTION_PRICE_ENTRY_MODE"
        ] = entry_mode

        globals()[
            "EXECUTION_PRICE_EXIT_MODE"
        ] = exit_mode

        globals()[
            "EXECUTION_PRICE_SEMANTICS_OK"
        ] = True

        globals()[
            "EXECUTION_PRICE_SEMANTICS_DECISION"
        ] = "PASS_EXECUTION_PRICE_SEMANTICS_FROZEN"


        # ----------------------------------------------------
        # 15. Final report
        # ----------------------------------------------------

        print()
        print("=" * 110)
        print("SELECTED EXECUTION PRICE CONTRACT")
        print("=" * 110)

        print(
            "ENTRY:",
            entry_mode
        )

        print(
            "EXIT:",
            exit_mode
        )

        print(
            "Return formula:",
            "SIGNED_SIMPLE_RETURN"
        )

        print(
            "Valid trades:",
            int(best["valid_rows"])
        )

        print(
            "Exact matches <= 1e-9:",
            int(best["matches_1e9"])
        )

        print(
            "Max gross-return difference:",
            float(best["max_abs_diff"])
        )

        print(
            "Mean gross-return difference:",
            float(best["mean_abs_diff"])
        )


        print()
        print("=" * 110)
        print("BOUNDARY CHECK")
        print("=" * 110)

        print(
            "Entry previous-close != current-open:",
            entry_boundary_diff
        )

        print(
            "Exit previous-close != current-open:",
            exit_boundary_diff
        )


        print()
        print("=" * 110)
        print("WORST REPRODUCTION DIFFERENCES")
        print("=" * 110)

        display(
            replay.assign(
                abs_diff=
                    replay["difference"].abs()
            )
            .sort_values(
                "abs_diff",
                ascending=False
            )
            .head(10)
        )


        print()
        print("=" * 110)
        print("FINAL DECISION")
        print("=" * 110)

        print(
            "EXECUTION PRICE SEMANTICS PASSED: True"
        )

        print(
            "FINAL DECISION: "
            "PASS_EXECUTION_PRICE_SEMANTICS_FROZEN"
        )

        print()
        print(
            "Champion/model/features were NOT modified."
        )

        print(
            "No orders were generated."
        )

        print()
        print("NEXT STEP:")
        print(
            "FINAL 2026 PAPER EXECUTION REPLAY "
            "WITH THE FROZEN ENTRY/EXIT PRICE CONTRACT."
        )


        return {
            "ok": True,
            "decision":
                "PASS_EXECUTION_PRICE_SEMANTICS_FROZEN",
            "entry_mode":
                entry_mode,
            "exit_mode":
                exit_mode,
            "max_diff":
                float(best["max_abs_diff"])
        }


    except Exception as e:

        print()
        print("=" * 110)
        print("DIAGNOSTIC STOPPED SAFELY")
        print("=" * 110)

        print(
            "Error:",
            type(e).__name__,
            ":",
            str(e)
        )

        print()
        print(
            "Champion was NOT modified."
        )

        print(
            "No order was generated."
        )

        globals()[
            "EXECUTION_PRICE_SEMANTICS_OK"
        ] = False

        globals()[
            "EXECUTION_PRICE_SEMANTICS_DECISION"
        ] = "STOP_IMPLEMENTATION_ERROR"

        return {
            "ok": False,
            "decision":
                "STOP_IMPLEMENTATION_ERROR",
            "error":
                str(e)
        }


# ============================================================
# RUN
# ============================================================

EXECUTION_PRICE_FINAL_REPORT = (
    run_one_shot_price_semantics()
)

print()
print("=" * 110)
print("CELL COMPLETE")
print("=" * 110)

print(
    "Decision:",
    EXECUTION_PRICE_FINAL_REPORT.get(
        "decision"
    )
)

print(
    "Passed:",
    EXECUTION_PRICE_FINAL_REPORT.get(
        "ok"
    )
)
